# Qlib 量化投资工作流教程

> **Colab 学生版 · 从数据到回测的完整实践**

本教程以沪深 300 为例，带你完成 Qlib 初始化、数据探索、Alpha158 特征构建、LightGBM 训练、组合回测与绩效分析。练习单元格保留为学生作业，其余部分可以在 Google Colab 云端运行。

**建议使用方式**

1. Notebook 会连接到固定的 Colab 2026.07 运行时（Python 3.12）。
2. 在顶部菜单选择 **代码执行程序 → 全部运行**。
3. 首次运行会安装依赖并下载约 464 MB 的教学数据，通常需要数分钟。
4. 教学展示只读取小时间段样本，并复用同一个 Alpha158 数据集，适配 Colab 标准内存运行时。
5. 遇到“请补充代码”的单元格时完成练习，再继续后续章节。

> 数据仅用于教学演示，不构成投资建议。Notebook 基于 [Microsoft Qlib v0.9.7 官方示例](https://github.com/microsoft/qlib/blob/v0.9.7/examples/workflow_by_code.ipynb) 改编，沿用 MIT License。

## 1. 云端环境准备

Colab 会为每位学习者提供临时 Python 环境，因此不需要在本地安装 Qlib。下面的初始化单元格会：

- 固定使用 Colab 2026.07 运行时（Python 3.12），避免 Qlib 与 Python 3.13 不兼容；
- 安装与本教程验证版本一致的 Qlib、Plotly、Statsmodels 和 LightGBM；
- 从 Qlib README 当前推荐的社区镜像下载 A 股教学数据；
- 将数据解压到 Colab 的 `/content/qlib_data/cn_data`；
- 检查 Python 版本和数据目录是否就绪。

> Colab 虚拟机是临时的。运行时被回收后，依赖与数据需要重新准备。
> 本教程采用低内存流程。不要把示例中的小样本改为三份全量数据同时常驻内存。

**连接故障快速判断**

如果在执行任何单元格之前就出现 `/api/kernelspecs` 500，请先新建一个空白 Colab 并运行 `print("ok")`。空白 Notebook 也失败，说明是 Colab 会话、账号配额或网络连接问题，不是本教程代码；请删除当前运行时、重新连接 CPU 运行时后再试。错误链接包含临时运行时令牌，不要公开转发。

In [ ]:
#@title 运行一次：安装依赖并准备 Qlib A 股数据
import os
import sys
import subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
SUPPORTED_PYTHON_MAX = (3, 12)
PINNED_PACKAGES = [
    "pyqlib==0.9.7",
    "plotly==6.6.0",
    "statsmodels==0.14.6",
    "lightgbm==4.6.0",
]

if IN_COLAB and sys.version_info[:2] > SUPPORTED_PYTHON_MAX:
    from IPython.display import HTML, display
    display(HTML("""
    <div style="padding:16px;border:2px solid #f59e0b;border-radius:12px;background:#fffbeb">
      <b>需要切换到 Python 3.12 运行时</b><br>
      当前 Colab 使用 Python 3.13，但 Qlib 0.9.7 尚未提供 Python 3.13 安装包。<br>
      请选择：<b>代码执行程序 → 更改运行时类型 → 运行时版本 → 2026.07</b>，保存后重新运行全部单元格。
    </div>
    """))
    raise RuntimeError(
        f"当前 Python {sys.version.split()[0]} 不受 Qlib 0.9.7 支持；请切换到 Colab 2026.07（Python 3.12）。"
    )

if IN_COLAB:
    print("正在安装教学环境……")
    install_result = subprocess.run(
        [sys.executable, "-m", "pip", "install", "--quiet", *PINNED_PACKAGES],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )
    if install_result.returncode != 0:
        print("\n依赖安装日志（最后 40 行）：")
        print("\n".join(install_result.stdout.splitlines()[-40:]))
        raise RuntimeError("教学环境安装失败，请保留上方日志并联系教师。")
    QLIB_DATA_DIR = Path("/content/qlib_data/cn_data")
else:
    print("当前不是 Colab：请先执行 pip install -r requirements-colab.txt")
    QLIB_DATA_DIR = Path("./qlib_data/cn_data").resolve()

calendar_file = QLIB_DATA_DIR / "calendars" / "day.txt"
if IN_COLAB and not calendar_file.exists():
    archive = Path("/content/qlib_bin.tar.gz")
    QLIB_DATA_DIR.mkdir(parents=True, exist_ok=True)
    print("正在下载 Qlib A 股教学数据（约 464 MB）……")
    subprocess.run(
        [
            "wget", "-q", "--show-progress",
            "https://github.com/chenditc/investment_data/releases/latest/download/qlib_bin.tar.gz",
            "-O", str(archive),
        ],
        check=True,
    )
    subprocess.run(
        ["tar", "-xzf", str(archive), "-C", str(QLIB_DATA_DIR), "--strip-components=1"],
        check=True,
    )
    archive.unlink(missing_ok=True)

if not calendar_file.exists():
    raise FileNotFoundError(
        f"没有找到 Qlib 数据：{QLIB_DATA_DIR}\n"
        "在 Colab 中请重新运行本单元格；本地运行请按 README 准备数据。"
    )

print(f"Python: {sys.version.split()[0]}")
print(f"Qlib 数据目录: {QLIB_DATA_DIR}")
print("环境准备完成 ✓")

In [ ]:
# 教程辅助函数（从 support/Utils_backtest.py 内嵌，确保 Colab 单文件可运行）
"""
回测结果分析模块

提供回测结果分析的完整功能，包括数据加载、回测报告分析、风险分析、模型性能分析等。
"""

import pandas as pd
import numpy as np
import plotly.graph_objects as go
from typing import Dict, Tuple, List
from qlib.contrib.report import analysis_position, analysis_model
from qlib.contrib.evaluate import risk_analysis
from qlib.contrib.eva.alpha import calc_ic
from scipy import stats as scipy_stats


# ==================== 工具函数 ====================

def calculate_drawdown_stats(returns: pd.Series) -> Dict:
    """计算回撤统计信息"""
    cum_returns = returns.cumsum()
    cum_max = cum_returns.cummax()
    drawdown = cum_returns - cum_max
    max_drawdown = drawdown.min()
    
    mdd_end = drawdown.idxmin()
    mdd_start = cum_returns.loc[:mdd_end].idxmax()
    
    if isinstance(mdd_start, str) and isinstance(mdd_end, str):
        from datetime import datetime
        start_date = datetime.strptime(mdd_start, '%Y-%m-%d')
        end_date = datetime.strptime(mdd_end, '%Y-%m-%d')
        duration = (end_date - start_date).days
    else:
        duration = (mdd_end - mdd_start).days if hasattr(mdd_end - mdd_start, 'days') else None
    
    return {
        '最大回撤': max_drawdown,
        '回撤起始日期': mdd_start,
        '回撤结束日期': mdd_end,
        '回撤持续天数': duration
    }


# ==================== 数据加载 ====================

def load_backtest_data(recorder, analysis_freq: str = "1day") -> Tuple[pd.DataFrame, Dict, pd.DataFrame]:
    """加载回测数据"""
    report_normal_df = recorder.load_object(f"portfolio_analysis/report_normal_{analysis_freq}.pkl")
    positions = recorder.load_object(f"portfolio_analysis/positions_normal_{analysis_freq}.pkl")
    analysis_df = recorder.load_object(f"portfolio_analysis/port_analysis_{analysis_freq}.pkl")
    return report_normal_df, positions, analysis_df


def get_backtest_data_info(report_normal_df: pd.DataFrame, positions: Dict, analysis_df: pd.DataFrame) -> Dict[str, pd.DataFrame]:
    """获取回测数据的基本信息"""
    report_info = pd.DataFrame({
        '属性': ['数据类型', '数据形状', '列名', '时间范围（开始）', '时间范围（结束）'],
        '值': [
            type(report_normal_df).__name__,
            f"{report_normal_df.shape[0]:,} × {report_normal_df.shape[1]}",
            ', '.join(report_normal_df.columns.tolist()),
            str(report_normal_df.index.min()),
            str(report_normal_df.index.max()),
        ]
    })
    report_info.set_index('属性', inplace=True)
    
    position_info = pd.DataFrame({
        '属性': ['数据类型', '持仓日期数量', '最早日期', '最晚日期'],
        '值': [
            type(positions).__name__,
            f"{len(positions):,}",
            str(min(positions.keys())),
            str(max(positions.keys())),
        ]
    })
    position_info.set_index('属性', inplace=True)
    
    analysis_info = pd.DataFrame({
        '属性': ['数据形状', '列名', '索引名称', '索引值数量'],
        '值': [
            f"{analysis_df.shape[0]:,} × {analysis_df.shape[1]}",
            ', '.join(analysis_df.columns.tolist()),
            ', '.join([str(name) for name in analysis_df.index.names if name is not None]) 
            if hasattr(analysis_df.index, 'names') and analysis_df.index.names 
            else str(type(analysis_df.index).__name__),
            f"{len(analysis_df.index):,}"
        ]
    })
    analysis_info.set_index('属性', inplace=True)
    
    return {'report_info': report_info, 'position_info': position_info, 'analysis_info': analysis_info}


# ==================== 回测报告分析 ====================

def extract_report_subplot(original_fig: go.Figure, yaxis_name: str, title: str, width: int = 1200, height: int = 600) -> go.Figure:
    """从report_graph的figure中提取指定子图"""
    subplot_traces = [trace for trace in original_fig.data if trace.yaxis == yaxis_name]
    fig = go.Figure(data=subplot_traces)
    fig.update_layout(
        width=width, height=height, title=title,
        xaxis=dict(type='category', tickangle=45, showline=True),
        yaxis=dict(zeroline=True, showline=True, showticklabels=True),
        hovermode='x unified',
        legend=dict(x=0.01, y=0.99, bordercolor="Black", borderwidth=1)
    )
    return fig


def analyze_cumulative_return(report_normal_df: pd.DataFrame) -> Dict[str, pd.DataFrame]:
    """分析累计收益"""
    cum_bench = report_normal_df['bench'].cumsum()
    cum_return_wo_cost = report_normal_df['return'].cumsum()
    cum_return_w_cost = (report_normal_df['return'] - report_normal_df['cost']).cumsum()
    
    cum_return_stats = pd.DataFrame({
        '指标': ['基准累计收益', '策略累计收益(不含成本)', '策略累计收益(含成本)', '交易成本影响'],
        '最终值': [cum_bench.iloc[-1], cum_return_wo_cost.iloc[-1], cum_return_w_cost.iloc[-1], 
                  cum_return_wo_cost.iloc[-1] - cum_return_w_cost.iloc[-1]],
        '百分比': [f"{cum_bench.iloc[-1]*100:.2f}%", f"{cum_return_wo_cost.iloc[-1]*100:.2f}%",
                  f"{cum_return_w_cost.iloc[-1]*100:.2f}%", f"{(cum_return_wo_cost.iloc[-1] - cum_return_w_cost.iloc[-1])*100:.2f}%"]
    })
    cum_return_stats.set_index('指标', inplace=True)
    return {'stats': cum_return_stats, 'cum_bench': cum_bench, 'cum_return_wo_cost': cum_return_wo_cost, 'cum_return_w_cost': cum_return_w_cost}


def analyze_drawdown(report_normal_df: pd.DataFrame, include_cost: bool = True) -> Dict:
    """分析回撤"""
    returns = report_normal_df['return'] - report_normal_df['cost'] if include_cost else report_normal_df['return']
    drawdown_stats = calculate_drawdown_stats(returns)
    drawdown_stats_df = pd.DataFrame({
        '指标': ['最大回撤', '回撤起始日期', '回撤结束日期', '回撤持续天数'],
        '数值': [drawdown_stats['最大回撤'], str(drawdown_stats['回撤起始日期']), 
                str(drawdown_stats['回撤结束日期']), f"{drawdown_stats['回撤持续天数']} 天" if drawdown_stats['回撤持续天数'] else "N/A"]
    })
    drawdown_stats_df.set_index('指标', inplace=True)
    return {'stats': drawdown_stats_df, 'drawdown_stats': drawdown_stats, 'label': '含成本' if include_cost else '不含成本'}


def compare_drawdown(report_normal_df: pd.DataFrame) -> pd.DataFrame:
    """对比含成本和不含成本的回撤"""
    dd_wo = analyze_drawdown(report_normal_df, include_cost=False)
    dd_w = analyze_drawdown(report_normal_df, include_cost=True)
    drawdown_comparison = pd.DataFrame({
        '指标': ['最大回撤', '回撤起始日期', '回撤结束日期', '回撤持续天数'],
        '不含成本': [f"{dd_wo['drawdown_stats']['最大回撤']:.4f}", str(dd_wo['drawdown_stats']['回撤起始日期']),
                    str(dd_wo['drawdown_stats']['回撤结束日期']), f"{dd_wo['drawdown_stats']['回撤持续天数']} 天" if dd_wo['drawdown_stats']['回撤持续天数'] else "N/A"],
        '含成本': [f"{dd_w['drawdown_stats']['最大回撤']:.4f}", str(dd_w['drawdown_stats']['回撤起始日期']),
                  str(dd_w['drawdown_stats']['回撤结束日期']), f"{dd_w['drawdown_stats']['回撤持续天数']} 天" if dd_w['drawdown_stats']['回撤持续天数'] else "N/A"],
        '差异': [f"{(dd_w['drawdown_stats']['最大回撤'] - dd_wo['drawdown_stats']['最大回撤']):.4f}", "-", "-",
                f"{(dd_w['drawdown_stats']['回撤持续天数'] - dd_wo['drawdown_stats']['回撤持续天数']) if (dd_w['drawdown_stats']['回撤持续天数'] and dd_wo['drawdown_stats']['回撤持续天数']) else 'N/A'} 天"]
    })
    drawdown_comparison.set_index('指标', inplace=True)
    return drawdown_comparison

def analyze_turnover(report_normal_df: pd.DataFrame) -> pd.DataFrame:
    """分析换手率"""
    turnover = report_normal_df['turnover']
    turnover_stats = pd.DataFrame({
        '指标': ['平均换手率', '换手率中位数', '换手率标准差', '最大换手率', '最小换手率', '年化换手率'],
        '数值': [turnover.mean(), turnover.median(), turnover.std(), turnover.max(), turnover.min(), turnover.mean() * 252],
        '百分比': [f"{turnover.mean()*100:.2f}%", f"{turnover.median()*100:.2f}%", f"{turnover.std()*100:.2f}%",
                  f"{turnover.max()*100:.2f}%", f"{turnover.min()*100:.2f}%", f"{turnover.mean() * 252:.2f}倍"]
    })
    turnover_stats.set_index('指标', inplace=True)
    return turnover_stats


def calculate_monthly_turnover(report_normal_df: pd.DataFrame) -> pd.DataFrame:
    """计算每个月的换手率统计"""
    turnover = report_normal_df['turnover']
    
    # 确保索引是datetime类型
    if not isinstance(turnover.index, pd.DatetimeIndex):
        turnover.index = pd.to_datetime(turnover.index)
    
    # 按年月分组计算月度换手率
    monthly_turnover = []
    for (year, month), group in turnover.groupby([turnover.index.year, turnover.index.month]):
        date_str = f"{year}-{month:02d}"
        monthly_turnover.append({
            'Date': date_str,
            'Year': year,
            'Month': month,
            'Mean Turnover': group.mean(),
            'Median Turnover': group.median(),
            'Std Turnover': group.std(),
            'Max Turnover': group.max(),
            'Min Turnover': group.min(),
            'Sum Turnover': group.sum(),
            'Trading Days': len(group)
        })
    
    monthly_df = pd.DataFrame(monthly_turnover)
    monthly_df.set_index('Date', inplace=True)
    return monthly_df


# ==================== 风险分析 ====================

def analyze_risk_indicators(analysis_df: pd.DataFrame, report_normal_df: pd.DataFrame, show_notebook: bool = True) -> Tuple[List, pd.DataFrame]:
    """分析风险指标"""
    risk_fig_list = analysis_position.risk_analysis_graph(analysis_df, report_normal_df, show_notebook=show_notebook)
    risk_data = analysis_df.unstack()
    risk_data.columns = risk_data.columns.droplevel(0)
    risk_summary = pd.DataFrame({
        '风险类型': ['超额收益（不含成本）', '超额收益（含成本）'],
        '标准差': [risk_data.loc['excess_return_without_cost', 'std'], risk_data.loc['excess_return_with_cost', 'std']],
        '年化收益率': [risk_data.loc['excess_return_without_cost', 'annualized_return'], risk_data.loc['excess_return_with_cost', 'annualized_return']],
        '信息比率': [risk_data.loc['excess_return_without_cost', 'information_ratio'], risk_data.loc['excess_return_with_cost', 'information_ratio']],
        '最大回撤': [risk_data.loc['excess_return_without_cost', 'max_drawdown'], risk_data.loc['excess_return_with_cost', 'max_drawdown']]
    })
    risk_summary.set_index('风险类型', inplace=True)
    return risk_fig_list, risk_summary


def calculate_monthly_risk_indicators(report_df: pd.DataFrame) -> Dict[str, pd.DataFrame]:
    """计算月度风险指标（三种类型对比）"""
    monthly_risks = {'无成本': [], '含成本': [], '基准': []}
    for (year, month), group in report_df.groupby([report_df.index.year, report_df.index.month]):
        if len(group) < 3:
            continue
        date_str = f"{year}-{month:02d}"
        excess_wo_cost = group['return'] - group['bench']
        risk_wo_cost = risk_analysis(excess_wo_cost, freq='day')
        monthly_risks['无成本'].append({
            'date': date_str, 'annualized_return': risk_wo_cost.loc['annualized_return', 'risk'],
            'information_ratio': risk_wo_cost.loc['information_ratio', 'risk'],
            'max_drawdown': risk_wo_cost.loc['max_drawdown', 'risk'], 'std': risk_wo_cost.loc['std', 'risk'],
        })
        excess_w_cost = group['return'] - group['bench'] - group['cost']
        risk_w_cost = risk_analysis(excess_w_cost, freq='day')
        monthly_risks['含成本'].append({
            'date': date_str, 'annualized_return': risk_w_cost.loc['annualized_return', 'risk'],
            'information_ratio': risk_w_cost.loc['information_ratio', 'risk'],
            'max_drawdown': risk_w_cost.loc['max_drawdown', 'risk'], 'std': risk_w_cost.loc['std', 'risk'],
        })
        bench_risk = risk_analysis(group['bench'], freq='day')
        monthly_risks['基准'].append({
            'date': date_str, 'annualized_return': bench_risk.loc['annualized_return', 'risk'],
            'information_ratio': bench_risk.loc['information_ratio', 'risk'],
            'max_drawdown': bench_risk.loc['max_drawdown', 'risk'], 'std': bench_risk.loc['std', 'risk'],
        })
    monthly_dfs = {}
    for risk_type, risk_list in monthly_risks.items():
        df = pd.DataFrame(risk_list)
        df.set_index('date', inplace=True)
        monthly_dfs[risk_type] = df
    return monthly_dfs


def analyze_monthly_risk(report_normal_df: pd.DataFrame, analysis_df: pd.DataFrame) -> Dict[str, pd.DataFrame]:
    """分析月度风险指标"""
    monthly_risk_dfs = calculate_monthly_risk_indicators(report_normal_df)
    ir_w_cost = monthly_risk_dfs['含成本']['information_ratio']
    best_month = ir_w_cost.idxmax()
    worst_month = ir_w_cost.idxmin()
    monthly_comparison = pd.DataFrame({
        '指标': ['信息比率', '年化收益率', '最大回撤', '标准差'],
        '最佳月份': [f"{monthly_risk_dfs['含成本'].loc[best_month, 'information_ratio']:.4f}",
                    f"{monthly_risk_dfs['含成本'].loc[best_month, 'annualized_return']:.4f}",
                    f"{monthly_risk_dfs['含成本'].loc[best_month, 'max_drawdown']:.4f}",
                    f"{monthly_risk_dfs['含成本'].loc[best_month, 'std']:.4f}"],
        '最差月份': [f"{monthly_risk_dfs['含成本'].loc[worst_month, 'information_ratio']:.4f}",
                    f"{monthly_risk_dfs['含成本'].loc[worst_month, 'annualized_return']:.4f}",
                    f"{monthly_risk_dfs['含成本'].loc[worst_month, 'max_drawdown']:.4f}",
                    f"{monthly_risk_dfs['含成本'].loc[worst_month, 'std']:.4f}"]
    })
    monthly_comparison['最佳月份名称'] = best_month
    monthly_comparison['最差月份名称'] = worst_month
    monthly_comparison.set_index('指标', inplace=True)
    
    stability_list = []
    for risk_type in ['无成本', '含成本', '基准']:
        df = monthly_risk_dfs[risk_type]
        ir = df['information_ratio']
        stability_list.append({
            '类型': risk_type, 'IR 均值': ir.mean(), 'IR 标准差': ir.std(),
            'IR 变异系数': ir.std() / ir.mean() if ir.mean() != 0 else 0,
            'IR > 0 的月份占比 (%)': (ir > 0).sum() / len(ir) * 100
        })
    stability_df = pd.DataFrame(stability_list)
    stability_df.set_index('类型', inplace=True)
    return {'monthly_dfs': monthly_risk_dfs, 'monthly_comparison': monthly_comparison, 'stability_analysis': stability_df}


def get_monthly_risk_summary(monthly_risk_dfs: Dict[str, pd.DataFrame]) -> Dict[str, pd.DataFrame]:
    """获取月度风险指标汇总"""
    metric_name_map = {'annualized_return': '年化收益率', 'information_ratio': '信息比率', 'max_drawdown': '最大回撤', 'std': '标准差'}
    summary_dict = {}
    for metric, metric_name in metric_name_map.items():
        metric_df = pd.DataFrame({
            '无成本': monthly_risk_dfs['无成本'][metric],
            '含成本': monthly_risk_dfs['含成本'][metric],
            '基准': monthly_risk_dfs['基准'][metric]
        })
        summary_dict[metric_name] = metric_df
    return summary_dict


# ==================== 模型性能分析 ====================

def analyze_ic(pred_label: pd.DataFrame, show_notebook: bool = True) -> Dict[str, pd.DataFrame]:
    """分析IC（信息系数）"""
    if show_notebook:
        analysis_position.score_ic_graph(pred_label)
    ic, rank_ic = calc_ic(pred_label['score'], pred_label['label'], date_col='datetime')
    
    ic_summary = pd.DataFrame({
        '指标类型': ['IC (Pearson)', 'IC (Pearson)', 'IC (Pearson)', 'IC (Pearson)',
                     'Rank IC (Spearman)', 'Rank IC (Spearman)', 'Rank IC (Spearman)', 'Rank IC (Spearman)'],
        '指标名称': ['均值', '标准差', 'ICIR (信息比率)', '胜率 (%)', '均值', '标准差', 'ICIR (信息比率)', '胜率 (%)'],
        '数值': [ic.mean(), ic.std(), ic.mean() / ic.std() if ic.std() > 0 else 0, (ic > 0).sum() / len(ic) * 100,
                rank_ic.mean(), rank_ic.std(), rank_ic.mean() / rank_ic.std() if rank_ic.std() > 0 else 0,
                (rank_ic > 0).sum() / len(rank_ic) * 100]
    })
    ic_summary_pivot = ic_summary.pivot_table(index='指标名称', columns='指标类型', values='数值')
    ic_summary_pivot = ic_summary_pivot.reindex(['均值', '标准差', 'ICIR (信息比率)', '胜率 (%)'])
    
    ic_dist_stats = pd.DataFrame({
        '统计指标': ['均值', '中位数', '标准差', '最小值', '最大值', '25%分位数', '75%分位数'],
        'IC (Pearson)': [ic.mean(), ic.median(), ic.std(), ic.min(), ic.max(), ic.quantile(0.25), ic.quantile(0.75)],
        'Rank IC (Spearman)': [rank_ic.mean(), rank_ic.median(), rank_ic.std(), rank_ic.min(), rank_ic.max(), rank_ic.quantile(0.25), rank_ic.quantile(0.75)]
    })
    ic_dist_stats.set_index('统计指标', inplace=True)
    
    ic_ts_stats = pd.DataFrame({
        '统计指标': ['均值', '中位数', '标准差', '最小值', '最大值', '25%分位数', '75%分位数', 'IC > 0 的占比 (%)'],
        'IC (Pearson)': [ic.mean(), ic.median(), ic.std(), ic.min(), ic.max(), ic.quantile(0.25), ic.quantile(0.75), (ic > 0).sum() / len(ic) * 100],
        'Rank IC (Spearman)': [rank_ic.mean(), rank_ic.median(), rank_ic.std(), rank_ic.min(), rank_ic.max(), rank_ic.quantile(0.25), rank_ic.quantile(0.75), (rank_ic > 0).sum() / len(rank_ic) * 100]
    })
    ic_ts_stats.set_index('统计指标', inplace=True)
    return {'ic': ic, 'rank_ic': rank_ic, 'summary': ic_summary_pivot, 'distribution': ic_dist_stats, 'time_series': ic_ts_stats}


def analyze_monthly_ic(ic: pd.Series) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """分析月度IC"""
    ic_df = pd.DataFrame({'IC': ic})
    ic_df.index = pd.to_datetime(ic_df.index)
    _index = ic_df.index.astype("str").str.replace("-", "").str.slice(0, 6)
    monthly_ic = ic_df['IC'].groupby(_index, group_keys=False).mean()
    monthly_ic.index = pd.MultiIndex.from_arrays([monthly_ic.index.str.slice(0, 4), monthly_ic.index.str.slice(4, 6)], names=["year", "month"])
    _month_list = pd.date_range(start=pd.Timestamp(f"{_index.min()[:4]}0101"), end=pd.Timestamp(f"{_index.max()[:4]}1231"), freq="ME")
    _years = [d.strftime("%Y%m%d")[:4] for d in _month_list]
    _months = [d.strftime("%Y%m%d")[4:6] for d in _month_list]
    fill_index = pd.MultiIndex.from_arrays([_years, _months], names=["year", "month"])
    monthly_ic_filled = monthly_ic.reindex(fill_index)
    monthly_ic_matrix = monthly_ic_filled.unstack()
    monthly_ic_summary = pd.DataFrame({
        '统计指标': ['月度IC均值', '月度IC标准差', '月度IC最小值', '月度IC最大值', 'IC > 0 的月份数', 'IC > 0 的月份占比 (%)'],
        '数值': [monthly_ic_filled.mean(), monthly_ic_filled.std(), monthly_ic_filled.min(), monthly_ic_filled.max(),
                (monthly_ic_filled > 0).sum(), (monthly_ic_filled > 0).sum() / len(monthly_ic_filled.dropna()) * 100]
    })
    monthly_ic_summary.set_index('统计指标', inplace=True)
    return monthly_ic_matrix, monthly_ic_summary


def analyze_ic_normality(ic: pd.Series, rank_ic: pd.Series) -> pd.DataFrame:
    """分析IC分布的正态性"""
    ic_shapiro = scipy_stats.shapiro(ic.dropna())
    rank_ic_shapiro = scipy_stats.shapiro(rank_ic.dropna())
    normality_test = pd.DataFrame({
        '检验方法': ['Shapiro-Wilk检验'],
        'IC (Pearson)': [f"统计量={ic_shapiro.statistic:.4f}, p值={ic_shapiro.pvalue:.4f}"],
        'Rank IC (Spearman)': [f"统计量={rank_ic_shapiro.statistic:.4f}, p值={rank_ic_shapiro.pvalue:.4f}"]
    })
    normality_test.set_index('检验方法', inplace=True)
    return normality_test


def analyze_group_returns(pred_label: pd.DataFrame, n_groups: int = 5, show_notebook: bool = True) -> Tuple[List, Dict[str, pd.DataFrame]]:
    """分析分组收益"""
    model_fig_list = analysis_model.model_performance_graph(pred_label, show_notebook=show_notebook)
    pred_label_sorted = pred_label.sort_values('score', ascending=False)
    pred_label_drop = pred_label_sorted.dropna(subset=['score'])
    group_returns = {}
    for i in range(n_groups):
        group_name = f'Group{i+1}'
        group_data = pred_label_drop.groupby(level='datetime', group_keys=False).apply(
            lambda x: x.iloc[len(x)//n_groups*i : len(x)//n_groups*(i+1)]['label'].mean()
        )
        group_returns[group_name] = group_data
    group_returns_df = pd.DataFrame(group_returns)
    group_returns_df.index = pd.to_datetime(group_returns_df.index)
    group_returns_df['long-short'] = group_returns_df['Group1'] - group_returns_df['Group5']
    avg_return = pred_label.groupby(level='datetime', group_keys=False)['label'].mean()
    group_returns_df['long-average'] = group_returns_df['Group1'] - avg_return
    cum_group_returns = group_returns_df.cumsum()
    
    group_cum_stats = pd.DataFrame({
        '分组': ['Group1 (最高分)', 'Group2', 'Group3', 'Group4', 'Group5 (最低分)', 'Long-Short', 'Long-Average'],
        '最终累计收益': [cum_group_returns['Group1'].iloc[-1], cum_group_returns['Group2'].iloc[-1], cum_group_returns['Group3'].iloc[-1],
                        cum_group_returns['Group4'].iloc[-1], cum_group_returns['Group5'].iloc[-1], cum_group_returns['long-short'].iloc[-1],
                        cum_group_returns['long-average'].iloc[-1]],
        '日均收益': [group_returns_df['Group1'].mean(), group_returns_df['Group2'].mean(), group_returns_df['Group3'].mean(),
                    group_returns_df['Group4'].mean(), group_returns_df['Group5'].mean(), group_returns_df['long-short'].mean(),
                    group_returns_df['long-average'].mean()],
        '收益标准差': [group_returns_df['Group1'].std(), group_returns_df['Group2'].std(), group_returns_df['Group3'].std(),
                      group_returns_df['Group4'].std(), group_returns_df['Group5'].std(), group_returns_df['long-short'].std(),
                      group_returns_df['long-average'].std()]
    })
    group_cum_stats.set_index('分组', inplace=True)
    
    dist_stats = pd.DataFrame({
        '指标': ['均值', '中位数', '标准差', '最小值', '最大值', '25%分位数', '75%分位数'],
        'Long-Short': [group_returns_df['long-short'].mean(), group_returns_df['long-short'].median(), group_returns_df['long-short'].std(),
                      group_returns_df['long-short'].min(), group_returns_df['long-short'].max(), group_returns_df['long-short'].quantile(0.25),
                      group_returns_df['long-short'].quantile(0.75)],
        'Long-Average': [group_returns_df['long-average'].mean(), group_returns_df['long-average'].median(), group_returns_df['long-average'].std(),
                        group_returns_df['long-average'].min(), group_returns_df['long-average'].max(), group_returns_df['long-average'].quantile(0.25),
                        group_returns_df['long-average'].quantile(0.75)]
    })
    dist_stats.set_index('指标', inplace=True)
    return model_fig_list, {'cumulative_stats': group_cum_stats, 'distribution_stats': dist_stats, 'group_returns_df': group_returns_df}


def analyze_pred_autocorr(pred_label_df: pd.DataFrame, lag: int = 1) -> pd.DataFrame:
    """分析预测自相关性"""
    pred = pred_label_df['score'].copy()
    autocorr_results = []
    for stock in pred.index.get_level_values('instrument').unique():
        stock_pred = pred.xs(stock, level='instrument')
        if len(stock_pred) > lag:
            autocorr = stock_pred.autocorr(lag=lag)
            if not pd.isna(autocorr):
                autocorr_results.append({'stock': stock, 'autocorr': autocorr, 'length': len(stock_pred)})
    return pd.DataFrame(autocorr_results)


def analyze_pred_autocorr_stats(autocorr_df: pd.DataFrame) -> Dict[str, pd.DataFrame]:
    """分析预测自相关性的统计信息"""
    autocorr_stats = pd.DataFrame({
        '统计指标': ['均值', '中位数', '标准差', '最小值', '最大值', '25%分位数', '75%分位数'],
        '自相关性': [autocorr_df['autocorr'].mean(), autocorr_df['autocorr'].median(), autocorr_df['autocorr'].std(),
                    autocorr_df['autocorr'].min(), autocorr_df['autocorr'].max(), autocorr_df['autocorr'].quantile(0.25),
                    autocorr_df['autocorr'].quantile(0.75)]
    })
    autocorr_stats.set_index('统计指标', inplace=True)
    high_autocorr = (autocorr_df['autocorr'] > 0.7).sum()
    medium_autocorr = ((autocorr_df['autocorr'] >= 0.3) & (autocorr_df['autocorr'] <= 0.7)).sum()
    low_autocorr = (autocorr_df['autocorr'] < 0.3).sum()
    total_stocks = len(autocorr_df)
    autocorr_category = pd.DataFrame({
        '自相关性水平': ['高自相关 (>0.7)', '中等自相关 (0.3-0.7)', '低自相关 (<0.3)'],
        '股票数量': [high_autocorr, medium_autocorr, low_autocorr],
        '占比 (%)': [high_autocorr / total_stocks * 100, medium_autocorr / total_stocks * 100, low_autocorr / total_stocks * 100]
    })
    autocorr_category.set_index('自相关性水平', inplace=True)
    return {'stats': autocorr_stats, 'category': autocorr_category}


# ==================== 图表生成辅助函数 ====================

def generate_report_graph(report_normal_df: pd.DataFrame, show_notebook: bool = True) -> List:
    """生成回测报告图表"""
    return analysis_position.report_graph(report_normal_df, show_notebook=show_notebook)


def show_monthly_risk_charts(risk_fig_list: List):
    """展示月度风险分析图表"""
    print("【月度风险分析图表1：年化收益率】")
    risk_fig_list[1].update_layout(width=1200, height=600, title='月度年化收益率')
    risk_fig_list[1].show()
    print("\n【月度风险分析图表2：最大回撤】")
    risk_fig_list[2].update_layout(width=1200, height=600, title='月度最大回撤')
    risk_fig_list[2].show()
    print("\n【月度风险分析图表3：信息比率】")
    risk_fig_list[3].update_layout(width=1200, height=600, title='月度信息比率')
    risk_fig_list[3].show()
    print("\n【月度风险分析图表4：标准差】")
    risk_fig_list[4].update_layout(width=1200, height=600, title='月度标准差')
    risk_fig_list[4].show()


def show_monthly_risk_summary(monthly_risk_dfs: Dict[str, pd.DataFrame]) -> Dict[str, pd.DataFrame]:
    """获取月度风险分析汇总（返回字典供notebook显示）"""
    metric_name_map = {'annualized_return': '年化收益率', 'information_ratio': '信息比率', 'max_drawdown': '最大回撤', 'std': '标准差'}
    summary_dict = {}
    for metric, metric_name in metric_name_map.items():
        metric_df = pd.DataFrame({
            '无成本': monthly_risk_dfs['无成本'][metric],
            '含成本': monthly_risk_dfs['含成本'][metric],
            '基准': monthly_risk_dfs['基准'][metric]
        })
        summary_dict[metric_name] = metric_df
    return summary_dict



In [ ]:
import sys, site
from pathlib import Path

In [ ]:
import qlib
import pandas as pd
from qlib.constant import REG_CN                                  # 导入中国区域常量
from qlib.utils import init_instance_by_config                    # 导入工具函数：配置初始化实例
from qlib.workflow import R                                       # 导入 workflow 记录对象 R
from qlib.workflow.record_temp import SignalRecord, PortAnaRecord
from qlib.backtest import get_exchange
from qlib.backtest.high_performance_ds import PandasQuote # 导入记录模板：信号记录和组合分析记录
from qlib.utils import flatten_dict                               # 导入字典扁平化工具函数，将嵌套字典转换为单层字典，便于记录参数 

## 2. 框架初始化

在这一部分，我们将初始化 Qlib 框架，设置数据路径和股票池。这是整个工作流的第一步。

**学习目标**：
- 理解如何初始化 Qlib 框架
- 理解股票池（market）和基准指数（benchmark）

**步骤**1: 设置数据存储路径 

In [ ]:
# provider_uri: 指定 qlib 数据的存储位置（与数据下载的target_dir一致）
provider_uri = str(QLIB_DATA_DIR)  # Colab 与本地共用

**步骤**2: 初始化 Qlib 系统

**概念说明**：
- `qlib.init()` 是 Qlib 的核心初始化函数，用于启动整个框架
- 需要指定数据存储路径（provider_uri）和市场区域（region） 
    - region：不同模式会导致不同的交易限制和成本。region 只是用于定义一批配置的快捷方式，包括最小交易单位（trade_unit）、交易限制（limit_threshold）等。它不是必需的，如果现有的 region 设置无法满足需求，用户可以手动设置关键配置。
- 初始化后，才能使用 Qlib 的各种功能

In [ ]:
# ==================== 初始化 Qlib 框架 ====================
# 启动 Qlib 框架，指定数据路径和市场区域

# 参数说明：
# provider_uri：数据存储路径，Qlib 会从这个路径读取股票数据
# region：市场区域，REG_CN 表示中国市场（A股）

mlflow_db = (Path("/content") if IN_COLAB else Path.cwd()) / "qlib_mlflow.db"
exp_manager = {
    "class": "MLflowExpManager",
    "module_path": "qlib.workflow.expm",
    "kwargs": {
        "uri": f"sqlite:///{mlflow_db.resolve().as_posix()}",
        "default_exp_name": "Experiment",
    },
}

qlib.init(provider_uri=provider_uri, region=REG_CN, exp_manager=exp_manager, kernels=1)
print(f"MLflow 实验数据库: {mlflow_db}")

**步骤3**: 指定股票池

In [ ]:
# ==================== 股票池和基准指数配置 ====================

market = "csi300"        # 股票池（指定要研究的股票集合）：沪深300成分股
benchmark = "SH000300"   # 基准指数（指定要对比的指数）：沪深300指数

## 3. 数据探索

在这一部分，我们将学习如何探索和了解 Qlib 中的股票数据。在开始正式的数据处理之前，先了解数据的结构对理解后续步骤很重要。

**学习目标**：
- 理解如何查看股票列表和特征数据
- 了解原始特征和派生特征的区别
- 理解动态成分股的概念（股票池会随时间变化）

In [ ]:
# D对象（"Data"缩写）是Qlib中的全局实例，为数据检索提供统一接口。调用qlib.init()后，D即可响应数据请求。
from qlib.data import D
from pprint import pprint

### 3.1 查看成分股列表

对于大部分指数来说，其成分股列表都不是一成不变的。因此，我们需要指定一个时间段，然后查看成分股列表在该时间段包含哪些公司。

In [ ]:
D.instruments('csi300')

In [ ]:
# 查询一年沪深指数成分股
instruments_dict = D.list_instruments(
    instruments=D.instruments('csi300'),
    start_time="2019-01-01",
    end_time="2019-12-31"
)
print('总数量', len(instruments_dict))

In [ ]:
# 使用 DataFrame 展示部分成分股及其生效区间
preview_items = list(instruments_dict.items())
rows = []
for inst, spans in preview_items:
    for span in spans:
        start_date = span[0]
        end_date = span[1]
        rows.append({
            'instrument': inst,
            'start_date': start_date,
            'end_date': end_date,
        })
preview_df = pd.DataFrame(rows)
display(preview_df)

In [ ]:
instruments_dict = D.list_instruments(
    instruments=D.instruments("csi300"),
    start_time="2019-02-04",
    end_time="2019-02-04",
    as_list=True # as_list=True：返回列表（list）；默认值as_list=False：返回字典（dict），键是股票代码，值是该股票在指定时间段内的“生效区间”
)
print(len(instruments_dict))

### 3.2 查看特征

#### 原始特征

In [ ]:
# ============================================
# 查看预置数据的全部原始特征
# ============================================

# 查看某个股票的可用特征
stock_features_path = QLIB_DATA_DIR / "features" / "sh600000"  # 以浦发银行为例

# 列出所有特征文件
feature_files = list(stock_features_path.glob("*.day.bin"))
features = [f.stem.split('.')[0] for f in feature_files]
print("可用的原始特征:")
for feature in sorted(features):
    print(f"  ${feature}", end=" ")

总结：qlib 中国市场数据（通过 `qlib-data` 下载的标准数据），**通常包含以下原始特征**：

In [ ]:
fields_basic = [
    "$open",      # 开盘价
    "$high",      # 最高价
    "$low",       # 最低价
    "$close",     # 收盘价
    "$volume",    # 成交量
    "$factor",    # 复权因子
    "$change",    # 涨跌额
]

**注意事项：**
1. 不同数据源可能提供不同的字段
2. 高频数据（如分钟线）可能有额外字段
3. 自定义数据可以包含任意字段

In [ ]:
# ============================================
# 示例1: 查看单个股票的单个特征
# ============================================
df1 = D.features(
    instruments=["SH600000"],  # 浦发银行
    fields=["$close"],         # 收盘价
    start_time="2020-01-01",
    end_time="2020-12-31"
)
display(df1.head(5))

In [ ]:
# ============================================
# 示例2: 查看多个股票的多个特征
# ============================================
df2 = D.features(
    instruments=["SH600000", "SH600016", "SH600019"],
    fields=fields_basic,
    start_time="2020-01-01",
    end_time="2020-01-03"
)
display(df2)

In [ ]:
# ============================================
# 示例3: 使用市场指数查看所有成分股
# ============================================
df3 = D.features(
    instruments=D.instruments("csi300"),  # CSI300所有成分股
    fields=["$close"],
    start_time="2020-01-01",
    end_time="2020-01-03"
)
print(f"数据形状: {df3.shape}")
print(f"涉及股票数: {len(df3.index.get_level_values('instrument').unique())}")
display(df3.head(10))

In [ ]:
# ============================================
# 示例4: 不指定时间范围（查看所有可用数据）
# ============================================
df4 = D.features(
    instruments=["SH600000"],
    fields=["$close"]
    # 不指定 start_time 和 end_time
)
print(f"数据起始: {df4.index.get_level_values('datetime').min()}")
print(f"数据结束: {df4.index.get_level_values('datetime').max()}")
print(f"总记录数: {len(df4)}")

#### 派生特征

**概念说明**：
- **派生特征**：基于原始特征计算得到的衍生指标
- 例如：移动平均线（MA）、相对强弱指标（RSI）等
- 这些特征通过数学运算从原始特征中提取，通常包含更多信息

In [ ]:
# ============================================
# 示例5: 使用表达式计算派生特征
# ============================================
df5 = D.features(
    instruments=["SH600000", "SH600016"],
    fields=[
        "$close",                                    # 原始收盘价
        "Ref($close, 1)",                            # 前一天收盘价
        "($close - Ref($close, 1)) / Ref($close, 1)",  # 日收益率
        "Mean($close, 5)",                           # 5日均价
        "Std($close, 20)",                           # 20日标准差
        # MACD 相关
        "EMA($close, 12)",                           # 12日指数移动平均
        "EMA($close, 26)",                           # 26日指数移动平均
        "EMA($close, 12) - EMA($close, 26)",         # MACD DIF
        # 布林带
        "Mean($close, 20) + 2 * Std($close, 20)",    # 上轨
        "Mean($close, 20) - 2 * Std($close, 20)",    # 下轨
        # 动量指标
        "$close / Ref($close, 10) - 1",              # 10日动量
    ],
    start_time="2020-01-01",
    end_time="2020-01-31"
    )
# 可以重命名列
df5.columns = ['close', 'close_lag1', 'return', 'ma5', 'std20', 'ema12', 'ema26', 'macd_dif', 
               'bb_upper', 'bb_lower', 'momentum10']
display(df5.head(5))

注意到`D.features`返回值的类型是pandas dataframe，因此dataframe的相关处理技巧也适用于`D.features`的返回值。

In [ ]:
# ============================================
# 示例6: 实际应用 - 计算相关性矩阵
# ============================================
df6 = D.features(
    instruments=D.instruments("csi300"),
    fields=["$close"],
    start_time="2019-01-01",
    end_time="2019-12-31"
)

# 转换为宽格式并计算收益率
df6 = df6.unstack(level='instrument')
display(df6.head())

returns = df6.pct_change(fill_method=None)
correlation_matrix = returns.corr()

print(f"相关性矩阵形状: {correlation_matrix.shape}")
display(correlation_matrix.iloc[:5, :5])  # 显示前5×5

更多可选运算符，可以阅读`qlib.data.ops`的帮助文档。

### **课后作业（1）：股票筛选**
参考链接: https://qlib.readthedocs.io/en/latest/reference/api.html#data

股票池：csi300
时间：2015-01-01 到 2016-02-15

1. 使用 NameDFilter 从中筛选出只在上交所（SH）上市的成分股（即代码以 SH 开头），计算占比

In [ ]:
# 请补充代码

2. 使用 ExpressionDFilter，筛选收盘价曾大于 1000 的股票

In [ ]:
# 请补充代码

3. 使用 ExpressionDFilter，筛选过去 60 日涨幅超过 50% 的股票

In [ ]:
# 请补充代码

## 4. 数据准备

在这一部分，我们将准备用于模型训练的数据。这是整个工作流的核心步骤。

**学习目标**
- 理解如何配置数据处理器（DataHandler）
- 理解如何配置数据集（DatasetH）
- 了解数据分割和归一化的基本概念

**DataHandler（数据处理器）**
- 负责从原始数据中提取特征和标签，并进行数据标准化

**DatasetH（数据集）**
- 基于 DataHandler 创建，负责数据分割（train/valid/test）和数据获取

**示例任务**
- 构造基于Alpha158特征集的数据集用于跨日收益预测模型训练

*Alpha158特征集*
- Qlib量化投资平台提供的一个经典特征集，包含158个经过精心设计的量化因子。这些因子主要基于股票的开盘价、最高价、最低价、收盘价和成交量等基础数据，通过人工特征工程方法提取而成
    - **KBAR特征**（9个）：K线形态特征，如 `($close-$open)/$open` 等
    - **PRICE特征**（4个）：原始价格特征，如 `$open/$close`, `$high/$close` 等
    - **ROLLING特征**（145个）：滚动统计特征，包含约30种技术指标
        - 使用 `windows=[5, 10, 20, 30, 60]` 天的5种滚动窗口计算
        - 包括趋势类指标（ROC, MA, BETA等）、波动性指标（STD, RSQR等）、极值指标（MAX, MIN等）、动量指标以及量价关系（CORR, CORD等）

*跨日收益率预测任务*
- 使用 T 日及之前的数据（特征）预测 T+1 → T+2 的收益率（标签）
    - 预测本身使用了 T 日的收盘价信息
    - 当天收盘后才能进行预测
    - 对应的策略需要在 T 日收盘后进行预测及分析，然后在 T+1 日开盘后进行交易，更符合实际交易场景
- 指标计算时间线图解

``` text
时间轴:
  T-2      T-1       T       T+1            T+2
   │        │        │       |              │
   │        │      今天       │              │
   │        │     (特征)      │              │
   │        │        │       │              │ 
   │        │        │     Close₁         Close₂
   │        │        │        ↓            ↓
   │        │        │    Ref($close,-1)  Ref($close,-2)
   │        │        │                ↓
   │        │        └────────────> 标签 = Close₂/Close₁ - 1
```


*数据集构造流程*
- 原始特征及标签提取 -> 数据标准化 -> 数据集划分

### 4.1 配置数据处理器（DataHandler）

数据处理器负责从原始数据中提取**特征**和**标签**，并进行**数据标准化**。我们使用 Qlib 提供的 `Alpha158` 数据处理器，它会自动构建 158 个特征以及对应跨日收益率标签

**步骤 1：配置参数**

首先，我们需要配置数据加载的参数。

In [ ]:
# ==================== 低内存教学配置 ====================
# 正式训练只覆盖 2017–2020；raw / infer / learn 教学另用一个月的小处理器。
# 正式处理器启用 drop_raw，只保留训练和预测所需的数据。
def show_process_memory(stage):
    """显示当前 Python 进程内存，便于区分内存终止与网络断连。"""
    try:
        import psutil
        rss_gb = psutil.Process().memory_info().rss / (1024 ** 3)
        print(f"[内存] {stage}: {rss_gb:.2f} GiB")
    except Exception:
        pass


TEACHING_SEGMENTS = {
    "train": ("2017-01-01", "2017-12-31"),
    "valid": ("2018-01-01", "2018-12-31"),
    "test": ("2019-01-01", "2020-08-01"),
}

# 免费 Colab 的组合回测只使用 2019 年。模型测试集仍保留到 2020-08，
# 但交易所行情缓存的时间范围更短，足以完成课堂演示和绩效分析。
LOW_MEMORY_BACKTEST_END = "2019-12-31"

demo_handler_config = {
    "start_time": "2019-01-01",
    "end_time": "2019-01-31",
    "fit_start_time": "2019-01-01",
    "fit_end_time": "2019-01-31",
    "instruments": market,
}

data_handler_config = {
    "start_time": TEACHING_SEGMENTS["train"][0],
    "end_time": TEACHING_SEGMENTS["test"][1],
    "fit_start_time": TEACHING_SEGMENTS["train"][0],
    "fit_end_time": TEACHING_SEGMENTS["train"][1],
    "instruments": market,
    "drop_raw": True,
}

print("低内存教学区间:", TEACHING_SEGMENTS)

**步骤 2：初始化数据处理器**

配置完成后，使用 `Alpha158` 初始化数据处理器。初始化时会自动：
- 加载原始数据
- 计算 158 个特征（价格、成交量、技术指标等）
- 生成标签（跨日收益率）
- 进行数据预处理（归一化、缺失值处理等）


In [ ]:
# ==================== 初始化一个月的演示处理器 ====================
from qlib.contrib.data.handler import Alpha158

# 这个小处理器只负责讲解因子与 raw / infer / learn，不参与正式训练。
demo_handler = Alpha158(**demo_handler_config)
handler = demo_handler
show_process_memory("一个月演示处理器初始化后")


#### Alpha158 数据处理器说明

`Alpha158` 是 Qlib 提供的预置数据处理器，它在初始化时会自动：

1. **构建 158 个特征**：通过 `Alpha158DL.get_feature_config()` 方法生成特征表达式配置 

2. **构建 1 个标签**：通过 `get_label_config()` 方法生成标签表达式
   - 默认标签表达式：`Ref($close, -2)/Ref($close, -1) - 1`

3. **数据预处理流程**：
   - 通过 `D.features` 计算表达式，生成原始数据（raw）
   - 在 `DataHandlerLP.process_data()` 方法中应用数据处理器：
     - **infer_processors=[]**：
       - 作用：处理特征数据，用于实际预测
       - Alpha158默认不处理特征（因为特征已在表达式层面相对化，如`KMID: ($close-$open)/$open`）
     - **learn_processors=[DropnaLabel, CSZScoreNorm]**：
       - 作用：处理标签数据，用于模型训练
       - `DropnaLabel`：删除标签为 NaN 的样本
       - `CSZScoreNorm`：对标签进行横截面标准化（只处理标签，不处理特征），即对每个交易日内的所有股票标签进行 Z-score 标准化，使每日标签均值为 0、标准差为 1。
   - 处理流程：`raw → shared_processors → infer_processors → _infer → learn_processors → _learn`
   - 生成三份数据：_data（原始数据 'raw'）、_infer（推理用处理后数据）、_learn（训练用处理后数据）



**理解 DataHandler 的三种数据模式**：

DataHandler 在初始化后会生成三种不同处理阶段的数据（raw、infer、learn），它们分别用于不同的场景。

**为什么需要区分这三种数据？**
- **训练时需要 learn 模式**：标签经过横截面标准化，便于模型学习相对强弱关系
- **预测时需要 infer 模式**：标签保持原始值，便于评估预测效果
- **查看原始数据 raw**：了解数据的原始分布和特征表达式

**为什么使用横截面标准化？**
1. **消除市场整体波动影响**：不同交易日市场整体涨跌不同，横截面标准化使每日标签分布一致
2. **便于模型训练**：标准化后的标签分布更稳定，有利于模型学习相对强弱关系
3. **保持相对排序**：标准化不改变股票间的相对排序，只调整分布

##### 特征(因子)查看

In [ ]:
# 从handler中获取因子配置
fields, names = handler.get_feature_config()

# 创建因子名称到表达式的映射
factor_dict = dict(zip(names, fields))

In [ ]:
# 查看特定因子
print(f"KMID: {factor_dict.get('KMID', '未找到')}")
print(f"MA5: {factor_dict.get('MA5', '未找到')}")

In [ ]:
# 查看所有因子的表达式
for name, expr in factor_dict.items():
    print(f"{name}: {expr}")

In [ ]:
# 获取一个月的特征样本，避免在教学展示阶段复制整套 Alpha158 数据
sample_period = slice("2019-01-01", "2019-01-31")
features = handler.fetch(selector=sample_period, col_set="feature")
print("特征样本形状:", features.shape)
features.head()

##### 标签查看

In [ ]:
# 查看 label 配置
print(handler.get_label_config())

In [ ]:
# 获取与特征相同时间段的标签样本
labels = handler.fetch(selector=sample_period, col_set="label")
print("标签样本形状:", labels.shape)
labels.head()

##### 数据预处理

In [ ]:
# ============================================
# 低内存方式：只比较一个月样本，不保留三份全量矩阵
# ============================================
import gc

sample_period = slice("2019-01-01", "2019-01-31")
mode_samples = {}

for data_key, label in (("raw", "原始"), ("infer", "推理"), ("learn", "学习")):
    sample = handler.fetch(selector=sample_period, data_key=data_key)
    mode_samples[data_key] = sample
    print(f"{label}数据样本形状: {sample.shape}")
    display(sample.head(2))

print("三种模式仅保留一个月样本；完整数据仍由 handler 统一管理。")

##### **课后作业（2）：理解三种数据模式（raw / infer / learn）**

下面的 `mode_samples` 来自独立的一个月演示处理器，足以比较三种模式；进入正式训练前会整体释放。

1. 比较三种样本的形状与缺失值数量；
2. 比较标签的均值和标准差；
3. 用文字说明 raw、infer、learn 分别适合什么场景。

In [ ]:
# 对比三种小样本的差异
print("数据形状对比:")
for key, frame in mode_samples.items():
    print(f"{key:>5}: {frame.shape}")

print("\n缺失值对比:")
# TODO: 请补充代码，分别计算三种样本的缺失值数量

print("\n标签统计对比:")
# TODO: 请补充代码，比较三种样本中 LABEL0 的均值与标准差

### 4.2 配置数据集（DatasetH）

数据集（DatasetH）将数据按时间分割成训练集、验证集和测试集。

In [ ]:
# ==================== 初始化数据集 ====================
# 先释放一个月的演示处理器，再创建启用 drop_raw 的正式处理器。
import gc

for variable_name in ["features", "labels", "mode_samples"]:
    globals().pop(variable_name, None)

del handler, demo_handler
gc.collect()

handler = Alpha158(**data_handler_config)
show_process_memory("正式 Alpha158 初始化后（drop_raw=True）")

# DatasetH 只引用这一个正式处理器，不会再复制一套 Alpha158。
from qlib.data.dataset import DatasetH

dataset = DatasetH(handler=handler, segments=TEACHING_SEGMENTS)

print("训练、验证、测试区间按时间顺序排列:")
for segment_name, segment_range in TEACHING_SEGMENTS.items():
    print(f"  {segment_name:>5}: {segment_range[0]} → {segment_range[1]}")

**DatasetH 的工作原理**：

`DatasetH` 是带数据处理器（DataHandler）的数据集类，作为数据处理器与模型训练之间的桥梁。

- 它通过 `segments` 字典管理数据分段（如 'train'、'valid'、'test'）
- 将数据预处理逻辑委托给内部的 `DataHandler` 实例
- 当调用 `prepare` 方法时，会根据传入的 segment 名称从 `segments` 中查找对应的时间范围
- 然后通过内部的 `_prepare_seg` 方法调用 `handler.fetch` 获取处理后的数据

这种设计将数据预处理（由 Handler 负责）与数据分段管理（由 DatasetH 负责）分离，使得用户可以通过简单的 segment 名称（如 `prepare('train')`）灵活获取不同阶段的数据。

In [ ]:
# ============================================
# 1. 查看训练期内一个月的小样本
# ============================================
sample_train_period = slice("2017-12-01", "2017-12-31")

train_learn_sample = dataset.prepare(sample_train_period, data_key="learn")
print("学习数据样本形状:", train_learn_sample.shape)
display(train_learn_sample.head())

train_infer_sample = dataset.prepare(sample_train_period, data_key="infer")
print("推理数据样本形状:", train_infer_sample.shape)
display(train_infer_sample.head())

del train_learn_sample, train_infer_sample
gc.collect()

In [ ]:
# ============================================
# 2. 只读取标签列检查时间段形状
# ============================================
segment_shapes = {}
for segment in ("train", "valid", "test"):
    segment_labels = dataset.prepare(segment, col_set="label")
    segment_shapes[segment] = (len(segment_labels), len(factor_dict))
    del segment_labels
    gc.collect()

print("\n数据集划分:")
print(f"训练集: {segment_shapes['train']}")
print(f"验证集: {segment_shapes['valid']}")
print(f"测试集: {segment_shapes['test']}")

In [ ]:
# ============================================
# 3. 使用小时间段查看特征和标签
# ============================================
sample_train_period = slice("2017-12-01", "2017-12-31")

features_df = dataset.prepare(sample_train_period, col_set="feature")
print("\n特征样本形状:", features_df.shape)
display(features_df.head())

label_df = dataset.prepare(sample_train_period, col_set="label")
print("\n标签样本形状:", label_df.shape)
display(label_df.head())

## 5. 训练预测模型

在这一部分，我们将使用机器学习模型来学习特征和标签之间的关系，从而预测股票的未来收益。

**学习目标**：
- 理解模型训练的基本流程
- 理解实验记录器（Recorder）的作用
- 掌握如何训练和保存模型

**示例任务**
- 使用LightGBM模型训练跨日收益预测模型，学习Alpha158特征与未来收益之间的关系

*模型选择：LightGBM（LGBModel）*
- LightGBM是微软开发的梯度提升决策树（GBDT）框架，具有训练速度快、内存占用低、准确率高等特点
- 本任务使用Qlib提供的`LGBModel`封装，支持自动早停等功能

*训练流程*
- 使用训练集（2008-2014）训练模型，学习特征与标签的映射关系
- 使用验证集（2015-2016）进行早停（Early Stopping），防止过拟合
- 模型训练完成后保存到实验记录器（Recorder），便于后续加载和使用

*实验记录*
- 使用Qlib的workflow模块记录训练过程，包括超参数、训练指标、模型文件等
- 便于实验对比、模型管理和结果复现

#### 5.1 模型和数据集配置

在这一部分，我们将配置LightGBM模型和数据集。

**配置说明**：

1. **模型（model）**：定义机器学习模型的类型和超参数
   - 本教程使用 `LGBModel`（LightGBM梯度提升树模型）
   - 超参数包括：学习率、树深度、正则化系数、采样比例等
   - 这些参数控制模型的复杂度和训练过程，影响模型的预测能力

2. **数据集（dataset）**：定义训练、验证和测试数据的时间范围
   - 使用之前配置的 `Alpha158` 数据处理器
   - 将数据分为三个时间段：训练集、验证集、测试集
   - 训练集用于模型学习，验证集用于早停防止过拟合，测试集用于最终评估

In [ ]:
# 使用配置字典定义模型和数据集参数，便于管理和复现
task = {
    "model": {
        "class": "LGBModel",
        "module_path": "qlib.contrib.model.gbdt",
        "kwargs": {
            "loss": "mse",                    # 均方误差损失函数
            "colsample_bytree": 0.8879,        # 特征采样比例（每棵树使用的特征比例）
            "learning_rate": 0.0421,           # 学习率（控制每轮迭代的步长）
            "subsample": 0.8789,               # 样本采样比例（每棵树使用的样本比例）
            "lambda_l1": 205.6999,             # L1 正则化系数（防止过拟合）
            "lambda_l2": 580.9768,             # L2 正则化系数（防止过拟合）
            "max_depth": 8,                    # 树的最大深度
            "num_leaves": 210,                 # 叶子节点数（控制模型复杂度）
            "num_threads": max(1, min(4, os.cpu_count() or 2)),                 # 线程数（并行训练）
        },
    },
    "dataset": {
        "class": "DatasetH",
        "module_path": "qlib.data.dataset",
        "kwargs": {
            "handler": {
                "class": "Alpha158",
                "module_path": "qlib.contrib.data.handler",
                "kwargs": data_handler_config,  # 使用之前配置的数据处理器
            },
            "segments": {
                "train": TEACHING_SEGMENTS["train"],  # 训练集：1年课堂轻量数据
                "valid": TEACHING_SEGMENTS["valid"],  # 验证集：1年数据（用于早停）
                "test": TEACHING_SEGMENTS["test"],   # 测试集：约1.5年数据（用于最终评估）
            },
        },
    },
}

# 使用配置初始化模型和数据集
model = init_instance_by_config(task["model"])
print("复用第 4 节已创建的 DatasetH，避免重复加载 Alpha158。")

#### 5.2 模型训练流程

在这一部分，我们将使用Qlib的workflow模块进行模型训练。训练过程包括：创建实验并记录配置参数、执行模型训练（自动早停）、保存训练好的模型，并获取实验ID用于后续加载和回测。

In [ ]:
# 训练前释放教学展示阶段的 DataFrame，降低 Colab 内存峰值
import gc

for variable_name in [
    "features", "labels", "mode_samples",
    "features_df", "label_df", "segment_shapes",
    "train_df", "valid_df", "test_df",
    "raw_data", "infer_data", "learn_data",
]:
    globals().pop(variable_name, None)

# 正式处理器已启用 drop_raw；保留兼容性检查，确保 raw 不再常驻。
if hasattr(handler, "_data"):
    del handler._data

gc.collect()
show_process_memory("训练前（已清理 raw 与演示变量）")

# 使用 Qlib workflow 记录训练实验
with R.start(experiment_name="train_model"):
    R.log_params(**flatten_dict(task))

    # 复用第 4 节创建的 dataset，不再初始化第二套 Alpha158
    model.fit(dataset)

    # 训练结束后释放 LightGBM 的训练矩阵，只保留可预测的 Booster
    if hasattr(model, "model") and hasattr(model.model, "free_dataset"):
        model.model.free_dataset()
    gc.collect()
    show_process_memory("模型训练后")

    R.save_objects(trained_model=model)
    rid = R.get_recorder().id

print(f"模型训练完成，Recorder ID: {rid}")

## 6. 预测、回测与分析

在这一部分，我们将使用训练好的模型进行预测，然后将预测结果转化为交易策略，并在历史数据上进行回测。

**学习目标**：
- 理解如何将模型预测转化为交易信号
- 了解交易策略的工作原理
- 掌握回测的基本流程
- 理解 SignalRecord 和 PortAnaRecord 的工作机制

**示例任务**
- 使用训练好的模型生成交易信号，执行TopKDropout选股策略，并在历史数据上进行回测分析

**交易策略：TopKDropoutStrategy**
- **TopK持仓**：每个交易日进行调仓，始终保持投资组合中有topk只股票（本任务中topk=50）
- **Dropout机制**：
  - **步骤1：计算买入候选股票池**
    - 从未持仓股票中，选择预测分数最高的候选股票
    - 候选池大小 = `n_drop + topk - 当前持仓数量`
  - **步骤2：确定卖出股票（防止"卖高买低"）**
    - 将当前持仓股票与买入候选股票池合并，按预测分数从高到低排序
    - 从合并列表的最后n_drop只中，找出属于当前持仓的股票
    - 这些股票即为需要卖出的（当前持仓中分数最低的，最多n_drop只）
    - **为什么要合并比较？** 确保不会卖出分数高于候选股票的持仓股票，避免"卖高买低"
  - **步骤3：确定买入股票**
    - 实际买入数量 = `卖出数量 + (topk - 当前持仓数量)`
    - 从候选股票池中选择分数最高的股票进行买入，确保调仓后持仓数量 = topk
- **固定换手率**：当持仓数量等于topk时，通过每天替换n_drop只股票，实现相对固定的换手率，便于控制交易成本
- **等权重分配**：对选中的股票进行等权重配置，简化资金管理
- **策略逻辑**：
  - 预测得分高的股票未来收益更高
  - 通过定期替换表现较差的股票，保持投资组合的质量
  - 卖出方式（method_sell）：默认"bottom"，卖出持仓中排名最差的股票
  - 买入方式（method_buy）：默认"top"，买入未持有中排名最好的股票
![TopkDropoutStrategy](https://fsj0621.github.io/images/topk_drop.png)

*回测执行流程*
- **加载模型**：从训练实验中获取已训练的模型
- **信号生成**：使用模型对测试集进行预测，得到每只股票在每个交易日的预测得分，SignalRecord 将预测结果保存为交易信号
- **策略执行**：TopKDropoutStrategy根据预测得分选择股票，生成交易订单
- **模拟交易**：SimulatorExecutor模拟交易执行过程，包括订单处理、成交价格、交易成本等
- **结果记录**：PortAnaRecord记录回测过程中的所有数据，包括持仓、收益、成本等

### 6.1 回测参数配置

在这一部分，我们将配置回测所需的参数，包括执行器、交易策略和回测参数。

**配置说明**：

1. **执行器（executor）**：负责模拟交易执行过程
   - 处理订单生成、成交模拟、持仓管理等
   - 模拟真实交易环境，包括交易成本、涨跌停限制等

2. **交易策略（strategy）**：定义如何根据模型预测进行选股和调仓
   - 本教程使用 `TopkDropoutStrategy`：始终保持 topk 只股票
   - 每个交易日淘汰表现最差的股票，买入排名靠前的股票

3. **回测参数（backtest）**：定义回测的基本设置
   - 回测时间范围、初始资金、基准指数
   - 交易规则：交易频率、涨跌幅限制、交易成本等

In [ ]:
port_analysis_config = {
    # ==================== 执行器配置 ====================
    # 执行器负责模拟交易执行过程，包括订单处理、成交模拟等
    "executor": {
        "class": "SimulatorExecutor",                    # 使用模拟执行器
        "module_path": "qlib.backtest.executor",         # 执行器模块路径
        "kwargs": {
            "time_per_step": "day",                      # 每步时间间隔：一天
            "generate_portfolio_metrics": True,          # 生成投资组合指标
        },
    },
    
    # ==================== 交易策略配置 ====================
    # TopkDropoutStrategy：基于排名的选股策略
    "strategy": {
        "class": "TopkDropoutStrategy",                  # 使用TopK淘汰策略
        "module_path": "qlib.contrib.strategy.signal_strategy",  # 策略模块路径
        "kwargs": {
            "signal": "<PRED>",                         # 复用 SignalRecord 已保存的预测
            "topk": 50,                                  # 选择前50只股票
            "n_drop": 5,                                 # 淘汰5只表现最差的股票
        },
    },
    
    # ==================== 回测参数配置 ====================
    # 定义回测的时间范围、初始资金、基准指数和交易规则
    "backtest": {
        "start_time": TEACHING_SEGMENTS["test"][0],                      # 回测开始时间
        "end_time": LOW_MEMORY_BACKTEST_END,                        # 回测结束时间
        "account": 100000000,                            # 初始资金：1亿元
        "benchmark": benchmark,                          # 基准指数（沪深300指数）
        "exchange_kwargs": {
            "codes": market,                              # 运行时会替换为预测中实际出现的股票
            "quote_cls": PandasQuote,                    # 避免 NumpyQuote 的二次 float64 缓存
            "freq": "day",                               # 交易频率：日频
            "limit_threshold": 0.095,                    # 涨跌幅限制阈值：9.5%
            "deal_price": "close",                       # 成交价格：收盘价
            "open_cost": 0.0005,                         # 开仓交易成本：0.05%
            "close_cost": 0.0015,                        # 平仓交易成本：0.15%
            "min_cost": 5,                               # 最小交易成本：5元
        },
    },
}

### 6.2 回测分析执行

在这一部分，我们将执行完整的回测流程：加载训练好的模型、生成交易信号、执行回测策略并生成分析报告。

#### 关键组件

- **SignalRecord**：模型预测 → 生成选股信号 → 保存 `pred.pkl` 和 `label.pkl`
- **PortAnaRecord**：加载预测信号 → 执行回测 → 风险与指标分析 → 保存回测报告、持仓、风险分析、指标分析等文件
- **数据传递机制**：两者通过共享同一个 `recorder` 实例来传递数据


#### SignalRecord.generate() - 生成预测信号

**作用**：使用模型对数据集进行预测，生成选股信号（股票得分）

**使用方式**：
```python
sr = SignalRecord(model, dataset, recorder)
sr.generate()
```

**内部执行流程**：

1. **调用模型预测**
   - 调用 `model.predict(dataset, segment="test")` 生成预测
   - 默认使用 `segment="test"`（对应 `dataset.segments` 中定义的测试集时间段）
   - 使用infer 模式，数据经过 `shared_processors + infer_processors` 处理

2. **生成预测结果**
   - 对测试集的所有交易日进行预测
   - 返回每只股票在每个交易日的预测分数（日频数据）
   - 如果返回 `pd.Series`，会自动转换为 `pd.DataFrame`，列名为 `"score"`

3. **保存预测结果到 recorder**
   - 将预测结果保存为 `pred.pkl`
   - 将真实标签保存为 `label.pkl`

#### PortAnaRecord.generate() - 执行回测分析

**作用**：根据预测信号执行回测，计算收益、风险等指标

**使用方式**：
```python
par = PortAnaRecord(recorder, port_analysis_config, rebalancing_freq)
par.generate()
```

**内部执行流程**：

1. **从 recorder 加载预测信号**
   - 加载 `pred.pkl`（由 `SignalRecord` 生成）

2. **配置执行器（Executor）**
   - 根据 `port_analysis_config` 创建执行器（如 `SimulatorExecutor`）
   - 设置调仓频率（`time_per_step`，如 `"day"`、`"month"`）

3. **执行回测循环**
   在每个调仓日：
   - 执行器触发调仓
   - 策略（Strategy）从 `pred.pkl` 中取出该日期的预测分数
   - 策略生成交易决策（如选择 topk 只股票）
   - 订单生成器（OrderGenerator）生成订单列表
   - 执行器执行订单，更新持仓

4. **生成回测报告**
   - 计算收益率、风险指标、最大回撤等
   - 保存持仓数据（`positions_normal_{freq}.pkl`）
   - 保存回测报告（`report_normal_{freq}.pkl`）
   - 保存投资组合分析（`port_analysis_{freq}.pkl`）
   - 保存指标分析（`indicator_analysis_{freq}.pkl`）

In [ ]:
# ==================== 低内存回测流程 ====================
# 关键原则：模型只预测一次；交易所只缓存预测中出现的股票；
# 组合回测直接读取 recorder 中的 pred.pkl。
print("[1/4] 准备生成测试集预测信号……", flush=True)
show_process_memory("信号生成前")

with R.start(experiment_name="backtest_analysis"):
    recorder = R.get_recorder()
    ba_rid = recorder.id

    # 复用上一节内存中的训练模型，避免再从 MLflow 反序列化一份模型。
    sr = SignalRecord(model, dataset, recorder)
    sr.generate()
    print("[2/4] pred.pkl 已保存；准备释放模型与 Alpha158 数据集……", flush=True)

    # 从预测结果提取实际出现的股票，避免交易所解析整个动态股票池。
    pred_for_scope = recorder.load_object("pred.pkl")
    backtest_codes = sorted(pred_for_scope.index.get_level_values("instrument").unique())
    del pred_for_scope

    # 后续 IC 分析只需要标签列。先保留这一列，然后释放完整特征数据。
    label_df = dataset.prepare("test", col_set="label")
    label_df.columns = ["label"]

    del sr
    for variable_name in ["model", "dataset", "handler", "task"]:
        globals().pop(variable_name, None)
    gc.collect()
    show_process_memory("信号生成后、组合回测前")

    # 先单独创建交易所，便于看到进度。PandasQuote 是 Qlib 官方实现，
    # 避免默认 NumpyQuote 再生成一套 float64 行情缓存。
    print(
        f"[3/4] 创建低内存交易所（{len(backtest_codes)} 只股票，"
        f"截至 {LOW_MEMORY_BACKTEST_END}）……",
        flush=True,
    )
    exchange_kwargs = dict(port_analysis_config["backtest"]["exchange_kwargs"])
    exchange_kwargs["codes"] = backtest_codes
    backtest_exchange = get_exchange(
        start_time=port_analysis_config["backtest"]["start_time"],
        end_time=port_analysis_config["backtest"]["end_time"],
        **exchange_kwargs,
    )
    port_analysis_config["backtest"]["exchange_kwargs"] = {"exchange": backtest_exchange}
    gc.collect()
    show_process_memory("低内存交易所创建后")

    # port_analysis_config 使用 <PRED> 占位符；PortAnaRecord 会从当前
    # recorder 读取 pred.pkl，不会再次调用模型预测或创建第二个交易所。
    print("[4/4] 开始组合回测……", flush=True)
    par = PortAnaRecord(recorder, port_analysis_config, "day")
    par.generate()
    del par, backtest_exchange
    gc.collect()
    show_process_memory("组合回测完成")

print(f"回测完成，Recorder ID: {ba_rid}")

## 7. 回测结果分析

在这一部分，我们将学习如何解读回测结果，理解各种指标的含义。

**学习目标**：
- 理解如何从 Recorder 中加载回测数据
- 理解各种回测指标的计算方法和含义
- 学会解读收益曲线和风险指标
- 掌握如何评估策略的表现

**分析内容**：
1. **数据准备与说明**：了解 Recorder 的作用，学习如何加载回测数据
2. **回测报告分析**：收益曲线、回撤分析、换手率等关键指标的可视化
3. **风险分析**：波动率、最大回撤、信息比率等风险指标的详细说明和分析
4. **模型性能分析**：IC（信息系数）分析，评估模型的预测能力

**注意：** Qlib中所有累计利润指标（如回报、最大回撤）均通过求和计算得出。 这样可以避免指标或图表随时间呈指数级偏斜。

### 7.1 数据准备与说明

在完成回测后，我们需要对结果进行深入分析，以评估策略的表现。qlib 提供了丰富的分析工具，帮助我们理解策略的各个方面。

**Recorder的作用**：
- `Recorder` 是 qlib 的实验记录器，基于 MLflow 实现
- 它保存了实验过程中的所有数据：预测结果、回测报告、持仓信息、风险分析等

In [ ]:
from qlib.contrib.report import analysis_model, analysis_position
from qlib.data import D
from qlib.workflow import R
# 回测分析辅助函数已在环境准备部分内嵌，无需额外文件。

# 从 recorder 加载数据
recorder = R.get_recorder(recorder_id=ba_rid, experiment_name="backtest_analysis")
print(recorder)

#### 预测数据
- 包含模型对每只股票在每个交易日的预测得分
- 索引为 MultiIndex (instrument, datetime)
- 用于计算 IC 等指标

In [ ]:
# ==================== 加载预测结果 ====================
pred_df = recorder.load_object("pred.pkl")

# 创建数据概览表格
pred_info = pd.DataFrame({
    '属性': [
        '数据形状',
        '列名',
        '索引名称',
        '日期范围（开始）',
        '日期范围（结束）',
        '股票数量'
    ],
    '值': [
        f"{pred_df.shape[0]:,} × {pred_df.shape[1]}",
        ', '.join(pred_df.columns.tolist()),
        ', '.join(pred_df.index.names),
        str(pred_df.index.get_level_values('datetime').min()),
        str(pred_df.index.get_level_values('datetime').max()),
        f"{len(pred_df.index.get_level_values('instrument').unique()):,}"
    ]
})
pred_info.set_index('属性')

print("\n【预测数据概览】")
display(pred_info)
print("\n【预测数据预览（前5行）】")
display(pred_df.head(5))

#### 回测报告数据
- 包含每个交易日的组合表现指标
- 主要列：`return`（组合收益率）、`cost`（交易成本）、`bench`（基准收益率）、`turnover`（换手率）
- 用于绘制收益曲线和风险分析

In [ ]:
# ==================== 加载回测数据 ====================
# 加载所有回测数据
report_normal_df, positions, analysis_df = load_backtest_data(recorder, analysis_freq="1day")

# 获取数据基本信息
data_info = get_backtest_data_info(report_normal_df, positions, analysis_df)

print("\n【回测报告信息】")
display(data_info['report_info'])
print("\n【回测报告预览（前5行）】")
display(report_normal_df.head(5))
print("\n【回测报告统计摘要】")
display(report_normal_df.describe().style.format('{:.4f}'))

#### 持仓信息数据
- 字典格式，键为日期，值为该日期的持仓详情
- 包含每只股票的持仓数量和权重

In [ ]:
# ==================== 持仓信息 ====================
print("\n【持仓信息概览】")
display(data_info['position_info'])

In [ ]:
# ==================== 查看持仓详情示例 ====================
# 选择一个日期，查看该日期的持仓情况
print("\n【持仓详情预览（示例：第一个日期）】")
sample_date = list(positions.keys())[10]
sample_pos = positions[sample_date]

if hasattr(sample_pos, 'get_stock_list'):
    stocks = sample_pos.get_stock_list()
    
    # 创建持仓信息表格
    if hasattr(sample_pos, 'get_stock_weight') and len(stocks) > 0:
        # 获取前10只股票的权重
        top_stocks = stocks[:10]
        position_data = {
            '股票代码': top_stocks,
            '持仓权重': [sample_pos.get_stock_weight(stock) for stock in top_stocks]
        }
        position_df = pd.DataFrame(position_data)
        position_df['持仓权重(%)'] = position_df['持仓权重'].apply(lambda x: f"{x*100:.2f}%")
        
        print(f"\n【{sample_date} 的持仓详情】")
        print(f"总持仓股票数量: {len(stocks)}")
        print(f"\n前10只股票持仓情况：")
        display(position_df[['股票代码', '持仓权重(%)']].style.format({
            '持仓权重': '{:.4f}'
        }))
    else:
        print(f"\n【{sample_date} 的持仓详情】")
        print(f"持仓股票数量: {len(stocks)}")
        print(f"前10只股票: {stocks[:10]}")

#### 风险分析数据
- 包含各种风险指标的汇总统计
- 用于生成风险分析图表

In [ ]:
# ==================== 风险分析数据 ====================
print("\n【风险分析数据概览】")
display(data_info['analysis_info'])

print("\n【风险分析数据预览】")
display(analysis_df)

### 7.2 回测报告分析

#### 7.2.1 回测报告概述

**什么是回测报告？**
- 回测报告包含每个交易日的策略表现数据
- 主要指标包括：收益率、交易成本、换手率等
- 通过这些数据，我们可以绘制收益曲线，分析策略表现

**回测报告的主要列**：
- `return`：策略收益率（不含成本）
- `cost`：交易成本
- `bench`：基准收益率
- `turnover`：换手率

#### 7.2.2 回测报告图表

In [ ]:
# 生成回测报告图表
analysis_position.report_graph(report_normal_df)

**说明**：`report_graph` 会生成一个包含7个子图的分析图表。下面我们将从这些图表中提取每个子图，单独展示并添加详细说明。

In [ ]:
# ==================== 获取原始figure ====================
if 'original_fig' not in locals():
    fig_list = generate_report_graph(report_normal_df, show_notebook=False)
    original_fig = fig_list[0]

In [ ]:
# fig_list

##### 子图1：累计收益曲线对比

**子图1：累计收益曲线对比 - 说明**

**图表说明**：
- **蓝色线（cum bench）**：基准累计收益曲线
- **橙色线（cum return w/o cost）**：策略累计收益曲线（不含交易成本）
- **绿色线（cum return w/ cost）**：策略累计收益曲线（含交易成本）

**计算公式**：
- 基准累计收益：$C_{bench}(t) = \sum_{i=1}^{t} r_{bench,i}$
- 策略累计收益（不含成本）：$C_{strategy}(t) = \sum_{i=1}^{t} r_{strategy,i}$
- 策略累计收益（含成本）：$C_{strategy}^{cost}(t) = \sum_{i=1}^{t} (r_{strategy,i} - c_i)$

**解读要点**：
1. **收益对比**：观察策略是否跑赢基准
2. **成本影响**：橙色线与绿色线的差距反映交易成本的影响
3. **趋势分析**：观察收益曲线的上升趋势和波动情况
4. **关键时点**：识别收益大幅波动的时间段


In [ ]:
# ==================== 子图1：累计收益曲线对比 ====================
# 提取子图
fig_subplot1 = extract_report_subplot(original_fig, yaxis_name='y', title='累计收益曲线对比', width=1200, height=600)
fig_subplot1.show()

In [ ]:
# ==================== 子图1：累计收益数值统计 ====================
# 分析累计收益
cum_return_result = analyze_cumulative_return(report_normal_df)
print("\n【累计收益统计】")
display(cum_return_result['stats'].style.format({'最终值': '{:.4f}'}))


In [ ]:
cum_return_result['cum_return_wo_cost']

##### 子图2：最大回撤分析（不含成本）

**子图2：最大回撤分析（不含成本） - 说明**

**图表说明**：
- 展示不考虑交易成本的回撤曲线
- 回撤曲线使用填充区域显示，便于观察回撤深度

**回撤的计算原理**：

回撤（Drawdown）衡量从历史最高点到当前点的下跌幅度：

$$
DD_t = C_t - H_t
$$

其中：
- $C_t = \sum_{i=1}^{t} r_i$ 为累计收益
- $H_t = \max_{j \in [1,t]} C_j$ 为历史最高点

最大回撤（MDD）为所有回撤中的最小值：

$$
MDD = \min_{t \in [1,T]} DD_t
$$

**解读要点**：
1. **回撤幅度**：观察回撤曲线的深度，MDD越小越好
2. **回撤持续时间**：从峰值到最低点的时间长度
3. **恢复能力**：观察回撤后的恢复速度


In [ ]:
# ==================== 子图2：最大回撤分析（不含成本） ====================
# 提取子图
fig_subplot2 = extract_report_subplot(original_fig, yaxis_name='y2', title='最大回撤分析（不含成本）', width=1200, height=600)
fig_subplot2.show()

In [ ]:
# ==================== 子图2：最大回撤数值统计（不含成本） ====================
# 分析回撤（不含成本）
dd_wo = analyze_drawdown(report_normal_df, include_cost=False)
print("\n【最大回撤统计（不含成本）】")
display(dd_wo['stats'].style.format({'数值': lambda x: f"{x:.4f}" if isinstance(x, (int, float)) else str(x)}))


##### 子图3：最大回撤分析（含成本）

In [ ]:
# ==================== 子图3：最大回撤分析（含成本） ====================
# 提取子图
fig_subplot3 = extract_report_subplot(original_fig, yaxis_name='y3', title='最大回撤分析（含成本）', width=1200, height=600)
fig_subplot3.show()

In [ ]:
# ==================== 子图3：最大回撤数值统计（含成本） ====================
# 分析回撤（含成本）
dd_w = analyze_drawdown(report_normal_df, include_cost=True)
print("\n【最大回撤统计（含成本）】")
display(dd_w['stats'].style.format({'数值': lambda x: f"{x:.4f}" if isinstance(x, (int, float)) else str(x)}))

In [ ]:
# ==================== 回撤对比 ====================
# 对比含成本和不含成本的回撤
drawdown_comparison = compare_drawdown(report_normal_df)
print("\n【最大回撤对比（不含成本 vs 含成本）】")
display(drawdown_comparison)


##### 子图4：超额收益曲线

**子图4：超额收益曲线 - 说明**

**图表说明**：
- **橙色线（不含成本）**：策略超额收益曲线（不考虑交易成本）
- **蓝色线（含成本）**：策略超额收益曲线（考虑交易成本）

**计算公式**：
- 超额收益（不含成本）：$R_{超额,t} = r_{策略,t} - r_{基准,t}$
- 超额收益（含成本）：$R_{超额,t}^{cost} = r_{策略,t} - r_{基准,t} - c_t$
- 累计超额收益：$C_{超额}(t) = \sum_{i=1}^{t} R_{超额,i}$

**解读要点**：
1. **超额收益趋势**：观察策略是否持续跑赢基准
2. **成本影响**：对比含成本和不含成本的超额收益差异
3. **波动性**：观察超额收益的波动情况
4. **稳定性**：观察超额收益是否稳定为正


In [ ]:
# ==================== 子图4：超额收益曲线 ====================
# 提取子图
fig_subplot4 = extract_report_subplot(original_fig, yaxis_name='y4', title='超额收益曲线', width=1200, height=600)
fig_subplot4.show()


##### **课后作业（3）：** 计算累计超额收益率

In [ ]:
def analyze_excess_return(report_normal_df: pd.DataFrame) -> pd.DataFrame:
    """课后作业：计算累计、均值与波动率等超额收益统计。"""
    # TODO: 请补充代码。返回一个以“指标”为索引、包含“数值”列的 DataFrame。
    return pd.DataFrame(columns=["数值"]).rename_axis("指标")

In [ ]:
# # ==================== 子图4：超额收益数值统计 ====================
# # 分析超额收益
# excess_return_stats = analyze_excess_return(report_normal_df)
# print("\n【超额收益统计】")
# display(excess_return_stats.style.format({
#     '数值': lambda x: f"{x:.6f}" if isinstance(x, (int, float)) else str(x)
# }))


##### 子图5：换手率

**子图5：换手率 - 说明**

**图表说明**：
- **蓝色曲线**：每日换手率变化

**换手率的计算公式**：

$$
Turnover_t = \frac{1}{2} \sum_{i} |w_{i,t} - w_{i,t-1}|
$$

其中：
- $w_{i,t}$ 为股票 $i$ 在第 $t$ 日的权重
- 除以 2 是因为买入和卖出都计算了权重变化，实际换手是总和的一半

**解读要点**：
1. **换手频率**：换手率越高，说明调仓越频繁
2. **成本影响**：高换手率会增加交易成本（年化换手率 = 平均日换手率 × 252）
3. **稳定性**：观察换手率的波动情况


In [ ]:
# ==================== 子图5：换手率 ====================
# 提取子图
fig_subplot5 = extract_report_subplot(original_fig, yaxis_name='y5', title='换手率曲线', width=1200, height=600)
fig_subplot5.show()

In [ ]:
# ==================== 子图5：换手率数值统计 ====================
# 分析换手率
turnover_stats = analyze_turnover(report_normal_df)
print("\n【换手率统计】")
display(turnover_stats.style.format({'数值': '{:.4f}'}))


##### 子图6：超额收益最大回撤数值统计（含成本）

**子图6：超额收益最大回撤（含成本） - 说明**

**图表说明**：
- 展示考虑交易成本后的超额收益回撤曲线

**解读要点**：
1. **超额收益回撤**：衡量策略相对基准的回撤情况
2. **成本影响**：观察成本对超额收益回撤的影响
3. **回撤恢复**：观察超额收益回撤后的恢复能力


In [ ]:
# ==================== 子图6：超额收益最大回撤（含成本） ====================
# 提取子图
fig_subplot6 = extract_report_subplot(original_fig, yaxis_name='y6', title='超额收益最大回撤（含成本）', width=1200, height=600)
fig_subplot6.show()

##### **课后作业（4）：** 计算超额收益最大回撤（含成本）

In [ ]:
def analyze_excess_return_drawdown(report_normal_df: pd.DataFrame) -> Dict[str, pd.DataFrame]:
    """课后作业：计算超额收益最大回撤及其起止日期。"""
    # TODO: 请补充代码。当前占位返回值保证“全部运行”不会出现语法错误。
    empty = pd.DataFrame(columns=["数值"]).rename_axis("指标")
    return {"without_cost": empty}

In [ ]:
# # ==================== 子图6：超额收益最大回撤数值统计（含成本） ====================
# # 分析超额收益回撤
# excess_dd_result = analyze_excess_return_drawdown(report_normal_df)
# print("\n【超额收益最大回撤统计（含成本）】")
# display(excess_dd_result['with_cost'].style.format({'数值': lambda x: f"{x:.6f}" if isinstance(x, (int, float)) else str(x)}))


##### 子图7：超额收益最大回撤（不含成本）

**子图7：超额收益最大回撤（不含成本） - 说明**

**图表说明**：
- 展示不考虑交易成本的超额收益回撤曲线
- 与含成本的超额收益回撤对比，可以观察成本对回撤的影响

**解读要点**：
1. **超额收益回撤**：衡量策略相对基准的回撤情况
2. **成本影响**：对比含成本和不含成本的超额收益回撤差异
3. **回撤恢复**：观察超额收益回撤后的恢复能力


In [ ]:
# ==================== 子图7：超额收益最大回撤（不含成本） ====================
# 提取子图
fig_subplot7 = extract_report_subplot(original_fig, yaxis_name='y7', title='超额收益最大回撤（不含成本）', width=1200, height=600)
fig_subplot7.show()

### 7.3 风险分析

**什么是风险分析？**
- 风险分析评估策略的风险水平
- 主要指标包括：波动率、最大回撤、信息比率等
- 好的策略不仅要收益高，还要风险可控

#### 7.3.1 风险指标说明

---

##### 标准差（std）

**定义**：衡量收益率序列的波动性，反映策略收益的稳定性。

**计算公式**：
$$\sigma = \sqrt{\text{Var}(r)} = \sqrt{\frac{1}{n-1}\sum_{i=1}^{n}(r_i - \bar{r})^2}$$

**参数说明**：
- $\sigma$：标准差
- $r_i$：第 $i$ 期的收益率
- $\bar{r}$：平均收益率，$\bar{r} = \frac{1}{n}\sum_{i=1}^{n}r_i$
- $n$：收益率序列的长度（样本数量）
- $\text{Var}(r)$：收益率序列的方差

---

##### 年化收益率（annualized_return）

**定义**：将日收益率转换为年化收益率，便于不同策略之间的比较。

**计算公式**：
$$R_{annual} = \bar{r} \times N$$

**参数说明**：
- $R_{annual}$：年化收益率
- $\bar{r}$：平均日收益率，$\bar{r} = \frac{1}{n}\sum_{i=1}^{n}r_i$
- $N$：年化因子
  - 日收益率：$N = 252$（一年约252个交易日）
  - 周收益率：$N = 52$（一年52周）
  - 月收益率：$N = 12$（一年12个月）

---

##### 信息比率（Information Ratio, IR）

**定义**：衡量超额收益与超额收益波动性的比值，反映策略在承担单位风险时获得的超额收益。

**计算公式**：
$$IR = \frac{\bar{R}_{excess}}{\sigma_{excess}} \times \sqrt{N}$$

**参数说明**：
- $IR$：信息比率
- $\bar{R}_{excess}$：平均超额收益率
  - 超额收益率 = 策略收益率 - 基准收益率
  - $\bar{R}_{excess} = \frac{1}{n}\sum_{i=1}^{n}(r_{strategy,i} - r_{benchmark,i})$
- $\sigma_{excess}$：超额收益的标准差（波动率）
  - $\sigma_{excess} = \sqrt{\frac{1}{n-1}\sum_{i=1}^{n}(r_{excess,i} - \bar{R}_{excess})^2}$
- $N$：年化因子（日收益率时 $N = 252$）
- $\sqrt{N}$：年化调整因子

---

##### 最大回撤（Maximum Drawdown, MDD）

**定义**：从历史最高点到最低点的最大跌幅，反映策略可能面临的最大损失。

**计算公式**：
$$MDD = \min_{t} (DD_t) = \min_{t} (C_t - H_t)$$

**参数说明**：
- $MDD$：最大回撤（通常为负值，绝对值越大表示回撤越大）
- $DD_t$：时刻 $t$ 的回撤值
- $C_t$：时刻 $t$ 的累计净值（累计收益）
- $H_t$：时刻 $t$ 之前的历史最高净值，$H_t = \max_{s \leq t} C_s$

**回撤率计算公式**：
$$DD_t = \frac{C_t - H_t}{H_t} = \frac{C_t}{H_t} - 1$$

#### 7.3.2 风险分析图表

In [ ]:
# 生成风险分析图表
analysis_position.risk_analysis_graph(analysis_df, report_normal_df)

**说明**：`risk_analysis_graph` 会生成一个包含5个子图的风险指标分析图表。下面我们将从这些图表中提取每个子图，单独展示并添加详细说明。

In [ ]:
# ==================== 获取风险分析图表（返回 figure 列表） ====================
# 获取风险分析图表
if 'risk_fig_list' not in locals():
    risk_fig_list, _ = analyze_risk_indicators(analysis_df, report_normal_df, show_notebook=False)

##### 风险指标对比图

**图表说明**：
风险指标对比图同时展示四个关键风险指标的对比（不含成本 vs 含成本）：
- **标准差（std）**：衡量收益率的波动性，值越大风险越高
- **年化收益率（annualized_return）**：将日收益率转换为年化收益率，便于比较
- **信息比率（information_ratio）**：衡量超额收益与风险的比值，IR 越高越好
- **最大回撤（max_drawdown）**：从历史最高点到最低点的最大跌幅，越小越好

**解读要点**：
1. **风险水平**：标准差和最大回撤反映策略的风险水平，值越小越好
2. **收益水平**：年化收益率反映策略的收益水平，值越大越好
3. **风险调整收益**：信息比率反映单位风险下的超额收益，IR > 1 表示策略表现良好，IR > 2 表示策略表现优秀
4. **成本影响**：对比含成本和不含成本的各项指标，观察交易成本对策略表现的影响

In [ ]:
# ==================== 风险指标对比图 ====================
# risk_fig_list[0] 同时展示四个风险指标：标准差、年化收益率、信息比率、最大回撤
# 每个指标都对比了不含成本和含成本两种情况
risk_fig_list[0].update_layout(width=1200, height=600, title='风险指标对比图')
risk_fig_list[0].show()


In [ ]:
# ==================== 风险指标对比图数值统计 ====================
# 分析风险指标
risk_fig_list, risk_summary = analyze_risk_indicators(analysis_df, report_normal_df, show_notebook=False)
print("\n【风险指标对比统计】")
display(risk_summary.style.format({
    '标准差': '{:.4f}',
    '年化收益率': '{:.4f}',
    '信息比率': '{:.4f}',
    '最大回撤': '{:.4f}'
}))


##### 月度风险分析图表

**图表说明**：
月度风险分析图表展示四个关键风险指标在每个月的变化趋势，帮助识别策略在不同时间段的表现：
- **月度年化收益率**：展示每个月的年化收益率变化趋势，可以观察策略在不同时间段的表现
- **月度最大回撤**：展示每个月的最大回撤变化趋势，可以观察策略在不同时间段的风险水平
- **月度信息比率**：展示每个月的信息比率变化趋势，可以观察策略在不同时间段的风险调整收益
- **月度标准差**：展示每个月的标准差变化趋势，可以观察策略在不同时间段的风险波动

**解读要点**：
1. **时间趋势**：观察各指标的时间变化趋势，识别策略表现较好或较差的时间段
2. **稳定性**：观察各指标的波动情况，评估策略的稳定性
3. **周期性**：识别是否存在周期性模式，如季节性效应
4. **风险控制**：结合多个指标，综合评估风险控制效果
5. **策略优化**：根据月度表现差异，识别需要优化的时间段或市场环境

In [ ]:
# ==================== 月度风险分析图表展示 ====================
# 展示月度风险分析图表
show_monthly_risk_charts(risk_fig_list)


In [ ]:
# ==================== 计算月度风险指标（三种类型对比） ====================
# 计算月度风险指标
# 如果monthly_risk_dfs不存在，则计算
if 'monthly_risk_dfs' not in locals():
    monthly_risk_result = analyze_monthly_risk(report_normal_df, analysis_df)
    monthly_risk_dfs = monthly_risk_result['monthly_dfs']

In [ ]:
# ==================== 月度表现对比（三种类型对比） ====================
# 获取月度风险分析汇总
monthly_summary = show_monthly_risk_summary(monthly_risk_dfs)

print("\n【月度风险分析汇总（三种类型对比）】")
print("=" * 100)
for metric_name, metric_df in monthly_summary.items():
    print(f"\n【{metric_name}】")
    print("-" * 80)
    display(metric_df.style.format('{:.4f}', na_rep=''))

In [ ]:
# ==================== 月度表现对比（最佳 vs 最差） ====================
# 获取月度风险分析结果（包含月度对比）
monthly_risk_result = analyze_monthly_risk(report_normal_df, analysis_df)
monthly_risk_dfs = monthly_risk_result['monthly_dfs']

print("\n【月度表现对比（最佳 vs 最差，基于含成本信息比率）】")
display(monthly_risk_result['monthly_comparison'])


In [ ]:
# ==================== 月度表现稳定性分析（三种类型对比） ====================
# 获取稳定性分析结果
print("\n【月度表现稳定性分析（三种类型对比）】")
display(monthly_risk_result['stability_analysis'].style.format({
    'IR 均值': '{:.4f}',
    'IR 标准差': '{:.4f}',
    'IR 变异系数': '{:.4f}',
    'IR > 0 的月份占比 (%)': '{:.2f}'
}))


### 7.4 模型性能分析

在这一部分，我们将分析模型的预测能力，评估模型是否有效。

**学习目标**：
- 理解 IC（信息系数）的含义和计算方法
- 学会评估模型的预测准确性
- 了解分组收益分析的方法
- 掌握模型性能评估的关键指标


In [ ]:
# 标签列已在信号生成后保留；完整 Alpha158 数据集已释放，避免再次抬高内存。
if "label_df" not in globals():
    raise RuntimeError("请先顺序运行第 6.2 节的低内存回测单元格。")

print(f"标签数据形状: {label_df.shape}")

In [ ]:
# 合并预测和标签
pred_label = pd.concat([label_df, pred_df], axis=1, sort=True).reindex(label_df.index)
print(f"预测标签数据形状: {pred_label.shape}")
display(pred_label.head())

#### 7.4.1 IC（信息系数）分析


**IC 的定义**：

IC（Information Coefficient，信息系数）衡量预测值与真实标签值的相关性，是评估模型预测能力的重要指标。

**为什么需要 IC？**

在量化投资中，我们不仅关心模型的预测准确度，更关心预测值能否有效区分股票的相对强弱。IC 正是衡量这种区分能力的指标：
- **IC > 0**：预测得分高的股票，真实收益也较高，模型能够有效识别优质股票
- **IC < 0**：预测得分高的股票，真实收益反而较低，模型预测方向错误
- **IC ≈ 0**：预测值与真实收益无关，模型无效

**Pearson IC（线性相关性）**：

对于每个交易日 $t$，计算预测值和标签值的 Pearson 相关系数：

$$
IC_t = \text{Corr}(\hat{y}_{i,t}, y_{i,t}) = \frac{\text{Cov}(\hat{y}_{i,t}, y_{i,t})}{\sigma_{\hat{y}} \cdot \sigma_y}
$$

其中：
- $\hat{y}_{i,t}$ 为股票 $i$ 在日期 $t$ 的预测值
- $y_{i,t}$ 为股票 $i$ 在日期 $t$ 的真实标签值
- $\text{Cov}$ 为协方差
- $\sigma$ 为标准差

Pearson IC 的特点：
- 衡量线性相关性，对异常值敏感
- 取值范围：$[-1, 1]$
- 适用于预测值和标签值都接近正态分布的情况

**Rank IC（秩相关性）**：

Rank IC 使用 Spearman 秩相关系数，对异常值更鲁棒：

$$
RankIC_t = \text{Corr}(\text{Rank}(\hat{y}_{i,t}), \text{Rank}(y_{i,t}))
$$

Rank IC 的特点：
- 只关心排序关系，不关心具体数值大小
- 对异常值不敏感，更稳健
- 在量化投资中更常用，因为选股策略主要依赖排序而非精确预测

**IC 统计指标**：

- **IC 均值**：$\bar{IC} = \frac{1}{T} \sum_{t=1}^{T} IC_t$
  - 衡量模型的平均预测能力
  - 通常要求 $\bar{IC} > 0.05$ 才认为模型有效
  
- **IC 标准差**：$\sigma_{IC} = \sqrt{\frac{1}{T-1} \sum_{t=1}^{T} (IC_t - \bar{IC})^2}$
  - 衡量 IC 的波动性
  - 标准差越小，模型预测越稳定
  
- **ICIR（IC 信息比率）**：$ICIR = \frac{\bar{IC}}{\sigma_{IC}}$
  - 类似于 Sharpe 比率，衡量风险调整后的预测能力
  - ICIR > 1 表示模型预测能力显著且稳定
  - 是评估模型质量的核心指标
  
- **IC 胜率**：$WinRate = \frac{1}{T} \sum_{t=1}^{T} \mathbf{1}(IC_t > 0) \times 100\%$
  - 表示 IC 为正的交易日占比
  - 胜率 > 50% 表示模型在多数时间有效

In [ ]:
# ==================== 生成IC图表 ====================
analysis_position.score_ic_graph(pred_label)

In [ ]:
# ==================== IC（信息系数）计算 ====================
# 分析IC
ic_result = analyze_ic(pred_label, show_notebook=False)
ic = ic_result['ic']
rank_ic = ic_result['rank_ic']

print("\n【IC 统计汇总】")
display(ic_result['summary'].style.format({
    'IC (Pearson)': lambda x: f'{x:.4f}' if abs(x) < 100 else f'{x:.2f}',
    'Rank IC (Spearman)': lambda x: f'{x:.4f}' if abs(x) < 100 else f'{x:.2f}'
}))

In [ ]:
# ==================== IC 分布统计 ====================
# 获取IC分布统计（已在analyze_ic中计算）
if 'ic_result' not in locals():
    ic_result = analyze_ic(pred_label, show_notebook=False)
    ic = ic_result['ic']
    rank_ic = ic_result['rank_ic']

print("\n【IC 分布统计】")
display(ic_result['distribution'].style.format('{:.4f}'))

#### 7.4.2 模型性能分析图表

In [ ]:
# ==================== 生成模型性能分析图表 ====================
analysis_model.model_performance_graph(pred_label)

In [ ]:
# ==================== 获取模型性能分析图表（返回 figure 列表） ====================
# 获取模型性能分析图表（已在Cell 140中生成，这里确保变量存在）
if 'model_fig_list' not in locals():
    model_fig_list, _ = analyze_group_returns(pred_label, n_groups=5, show_notebook=False)

##### **课后作业（5）：** 请按照自己的理解补充各图表解读

##### 分组收益累计曲线

**图表说明**：
分组收益累计曲线展示5个分组的累计收益变化趋势，直观反映模型的预测能力。

**计算公式**：
- 分组收益：$R_{k,t} = \frac{1}{|\text{Group}_k|} \sum_{i \in \text{Group}_k} y_{i,t}$
- 累计收益：$C_{k}(t) = \sum_{i=1}^{t} R_{k,i}$

**解读要点**：
1. **收益单调性**：理想情况下，Group1（最高分）的累计收益应该最高，Group5（最低分）的累计收益应该最低
2. **收益差距**：Group1 和 Group5 的累计收益差距越大，说明模型预测能力越强

In [ ]:
# ==================== 分组收益累计曲线 ====================
# model_fig_list[0] 是分组收益累计曲线图

model_fig_list[0].update_layout(width=1200, height=600, title='分组收益累计曲线')
model_fig_list[0].show()


In [ ]:
# ==================== 分组收益累计曲线数值统计 ====================
# 分析分组收益
model_fig_list, group_stats = analyze_group_returns(pred_label, n_groups=5, show_notebook=False)

print("\n【分组收益累计统计（与图表完全对应）】")
print("注：此表格包含图表中显示的所有分组（Group1~5、Long-Short、Long-Average）")
display(group_stats['cumulative_stats'].style.format({
    '最终累计收益': '{:.4f}',
    '日均收益': '{:.6f}',
    '收益标准差': '{:.6f}'
}))


##### 分组收益分布图

**图表说明**：
分组收益分布图展示 Long-Short（高分组-低分组）和 Long-Average（高分组-市场平均）的收益分布特征。

**计算公式**：
- Long-Short 收益：$R_{LS,t} = R_{Group1,t} - R_{Group5,t}$
- Long-Average 收益：$R_{LA,t} = R_{Group1,t} - \bar{R}_t$，其中 $\bar{R}_t = \frac{1}{|I|} \sum_{i \in I} y_{i,t}$

In [ ]:
# ==================== 分组收益分布图 ====================
# model_fig_list[1] 是分组收益分布图（包含 Long-Short 和 Long-Average 两个子图）
model_fig_list[1].update_layout(width=1200, height=600, title='分组收益分布图')
model_fig_list[1].show()


In [ ]:
# ==================== 分组收益分布数值统计 ====================
# 获取分组收益分布统计
print("\n【分组收益分布统计】")
display(group_stats['distribution_stats'].style.format('{:.6f}'))


##### IC柱状图

**图表说明**：
IC柱状图展示每日IC值的变化，可以观察模型预测能力的稳定性。

**计算公式**：
- IC：$IC_t = \text{Corr}(\hat{y}_{i,t}, y_{i,t})$
- Rank IC：$RankIC_t = \text{Corr}(\text{Rank}(\hat{y}_{i,t}), \text{Rank}(y_{i,t}))$


In [ ]:
# ==================== IC柱状图 ====================
# model_fig_list[2] 是IC柱状图

model_fig_list[2].update_layout(width=1200, height=600, title='IC柱状图')
model_fig_list[2].show()


In [ ]:
# ==================== IC柱状图数值统计 ====================
# 获取IC时间序列统计
print("\n【IC时间序列统计】")
display(ic_result['time_series'].style.format({
    'IC (Pearson)': lambda x: f'{x:.4f}' if abs(x) < 100 else f'{x:.2f}',
    'Rank IC (Spearman)': lambda x: f'{x:.4f}' if abs(x) < 100 else f'{x:.2f}'
}))


##### 月度IC热力图

**图表说明**：
月度IC热力图展示各月份的IC表现，可以识别模型在不同时间段的表现差异。

**计算公式**：
- 月度IC：$\bar{IC}_m = \frac{1}{|D_m|} \sum_{t \in D_m} IC_t$，其中 $D_m$ 为第 $m$ 个月的所有交易日


In [ ]:
# ==================== 月度IC热力图 ====================
# model_fig_list[3] 是月度IC热力图

model_fig_list[3].update_layout(width=1200, height=600, title='月度IC热力图')
model_fig_list[3].show()

In [ ]:
# ==================== 月度IC热力图数值统计 ====================
# 分析月度IC
monthly_ic_matrix, monthly_ic_summary = analyze_monthly_ic(ic)

print("\n【月度IC热力图数据（年份 x 月份矩阵）】")
display(monthly_ic_matrix.style.format('{:.4f}', na_rep=''))

print("\n【月度IC汇总统计】")
display(monthly_ic_summary.style.format({
    '数值': lambda x: f'{x:.4f}' if abs(x) < 100 else f'{x:.2f}'
}))


##### IC分布和Q-Q图

**图表说明**：
IC分布和Q-Q图用于分析IC值的分布特征和正态性。

**IC分布图**：
- 展示IC值的直方图分布
- 可以观察IC值是否接近正态分布

**Q-Q图（分位数-分位数图）**：
- 将IC值的分位数与理论正态分布的分位数进行对比
- 如果点大致落在一条直线上，说明IC值接近正态分布
- 偏离直线表示分布偏离正态分布


In [ ]:
# ==================== IC分布和Q-Q图 ====================
# model_fig_list[4] 是IC分布和Q-Q图（包含2个子图）

model_fig_list[4].update_layout(width=1200, height=600, title='IC分布和Q-Q图')
model_fig_list[4].show()


In [ ]:
# ==================== IC分布和Q-Q图数值统计 ====================
# 分析IC分布的正态性
ic_normality = analyze_ic_normality(ic, rank_ic)

print("\n【IC分布正态性检验】")
print("注：p值 > 0.05 表示不能拒绝正态分布假设")
display(ic_normality)

# IC分布统计（已在之前计算，这里显示汇总）
print("\n【IC分布统计汇总】")
display(ic_result['distribution'].style.format('{:.4f}'))


##### 预测自相关性图

**图表说明**：
预测自相关性图展示模型预测值在时间上的相关性，评估预测的稳定性。

**计算公式**：
- 自相关性：$AC_t = \text{Corr}(\text{Rank}(\hat{y}_{i,t}), \text{Rank}(\hat{y}_{i,t-1}))$


In [ ]:
# ==================== 预测自相关性图 ====================
# model_fig_list[5] 是预测自相关性图

model_fig_list[5].update_layout(width=1200, height=600, title='预测自相关性图')
model_fig_list[5].show()


In [ ]:
# ==================== 预测自相关性图数值统计 ====================
# 分析预测自相关性
autocorr_results = analyze_pred_autocorr(pred_label, lag=1)
autocorr_result = analyze_pred_autocorr_stats(autocorr_results)

print("\n【预测自相关性统计】")
display(autocorr_result['stats'].style.format('{:.6f}'))

print("\n【预测自相关性分类统计】")
display(autocorr_result['category'].style.format({
    '股票数量': '{:.0f}',
    '占比 (%)': '{:.2f}'
}))
